# 16 · Interpretability, Robustness and Uncertainty

**Part VII — Engineering, trust and practice** · Estimated time: ~120 min

## Learning objectives

- Explain why accuracy on a test set is not enough to trust a deployed model, using documented failure cases.
- Implement gradient saliency, SmoothGrad, Integrated Gradients, occlusion and Grad-CAM from scratch, and run a randomization sanity check on them.
- Visualize what individual channels and classes respond to with activation maximization, and measure how linearly decodable the label is at each depth with linear probes.
- Derive and implement FGSM and PGD attacks (untargeted and targeted), demonstrate transferability, and train a PGD-robust model while measuring the robustness–accuracy trade-off.
- Quantify the effect of distribution shift and detect out-of-distribution inputs with max-softmax and energy scores, evaluated by AUROC.
- Measure calibration with reliability diagrams and expected calibration error, and improve it with temperature scaling, MC dropout and deep ensembles.
- Compute demographic parity and equalized odds on a tabular dataset, apply a post-processing mitigation, and describe differential privacy and model documentation practices.

## Prerequisites

- Notebook 04 · Training Deep Networks: Optimization, Initialization, Normalization, Regularization (dropout, regularization).
- Notebook 05 · The PyTorch Workflow and Best Practices (training loops, evaluation, hooks).
- Notebook 06 · Convolutional Neural Networks (the CNN we train here; occlusion maps).
- Notebook 07 · Transfer Learning and Modern Vision (Grad-CAM with hooks, which we recap compactly).

## Contents

1. Why trust matters
2. Attribution methods from scratch
3. Feature visualization and probing representations
4. Adversarial examples and adversarial training
5. Distribution shift and out-of-distribution detection
6. Uncertainty and calibration
7. Fairness, privacy and ethics

In [ ]:
import math, os, random, time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import torchvision
from torchvision import datasets
from torchvision.transforms import InterpolationMode
from torchvision.transforms.v2 import functional as TF
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.linear_model import LogisticRegression
from scipy.stats import rankdata

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch {torch.__version__} | torchvision {torchvision.__version__} | device: {device}"
      + (f" ({torch.cuda.get_device_name(0)})" if device.type == "cuda" else ""))

# FAST_RUN = True shrinks every training budget so the whole notebook runs in a few
# minutes on a CPU. Set it to False for the full experiments (recommended on a GPU).
FAST_RUN = False
DATA_DIR = Path("../data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams["figure.figsize"] = (7, 4)
plt.rcParams["axes.grid"] = True

## 1. Why trust matters

A model that scores 99% on its test set has demonstrated one thing: on inputs drawn from the same distribution as the test set, its predictions agree with the labels. Deployment asks a longer list of questions. *Why* did it predict what it predicted, and is the reason one we would accept? What happens on inputs that look nothing like the training data, or on inputs crafted by someone who wants the model to fail? Does the model know when it does not know? Does it treat groups of people differently? Each of these questions has been answered badly by a real system, and it is worth starting with a few documented cases, because they set the agenda for the rest of the notebook.

**The right answer for the wrong reason.** Ribeiro, Singh and Guestrin (2016) trained a classifier to distinguish huskies from wolves and found, using their LIME explanations, that it had learned to detect *snow* in the background, because most wolf photos in the training set had been taken in winter. Zech et al. (2018) showed that a CNN for detecting pneumonia on chest X-rays, trained on data from several hospitals, was partly recognizing *which hospital* an image came from (from laterality markers and image-processing signatures) and using the hospital's base rate of pneumonia as a shortcut; its performance dropped sharply on images from a hospital it had not seen. These are instances of **shortcut learning**: the training distribution contains a feature that is easier to exploit than the intended one, and gradient descent takes the easy path. Section 2 develops tools for asking a model *where it is looking*.

**Brittleness under small changes.** Szegedy et al. (2014) discovered that imperceptible perturbations of an image can flip the prediction of a state-of-the-art classifier with high confidence, and Eykholt et al. (2018) showed that a few printed stickers on a physical stop sign could make a detector read it as a speed-limit sign from most viewpoints. Section 4 explains why such **adversarial examples** exist, constructs them, and trains a model that resists them.

**Silent failure under distribution shift.** Models are routinely confident on inputs they have never seen anything like: a digit classifier will happily assign 99% to a class when shown a photograph of a shoe. Medical imaging models trained at one site degrade at another; a model trained before a change in a data pipeline keeps producing confident outputs after it. Sections 5 and 6 quantify this and build detectors and calibrated uncertainty estimates.

**Unequal performance across groups.** Buolamwini and Gebru (2018) evaluated commercial face-analysis systems and found error rates of under 1% for lighter-skinned men and up to 35% for darker-skinned women, a disparity traced to unrepresentative training data. The COMPAS recidivism-risk debate (Angwin et al., ProPublica, 2016, and the subsequent statistical replies) showed that reasonable-sounding fairness criteria can be mutually incompatible when base rates differ between groups. Section 7 implements the two most common criteria and shows the trade-off.

None of these systems failed on their test sets. The common thread is that the test set answers a narrower question than the one deployment asks, and the methods in this notebook are ways of asking the wider questions *before* the users do.

> **Key idea.** Interpretability tells us *why* a model predicts, robustness tells us *when* the prediction can be changed, uncertainty tells us *how much* to trust it, and fairness tells us *for whom* it works. All four must be measured; none is implied by test accuracy.

## 2. Attribution methods from scratch

### 2.1 The running example: a small CNN on MNIST

Every method in this notebook is applied to one model, a small convolutional network trained on MNIST, so that we can compare the methods on the same predictions. The network takes images in $[0, 1]$ and normalizes them *inside* `forward`, which keeps every attribution, perturbation and noise level below in natural pixel units. It has three convolutional stages (the last one produces a $64 \times 14 \times 14$ map before pooling, which Grad-CAM will use), a hidden fully-connected layer with dropout (which Section 6 will reuse for MC dropout), and a ten-way output.

We load MNIST once as tensors: the raw `uint8` images become float tensors in $[0, 1]$, which is faster than applying a transform pipeline per image and lets us build shifted and noisy versions of the whole test set with a single tensor expression later. The training set is split into a training part and a held-out **validation** part; the validation part is reserved for fitting the temperature in Section 6 and must not be touched by training. Under `FAST_RUN` we use 6,000 training images, 2,000 validation and 2,000 test images and a single epoch, which gives roughly 90% accuracy (enough for every experiment below to show its effect, although the maps and curves are cleaner with a better model); the full run uses the whole dataset and four epochs, which gives about 99%.

In [ ]:
def load_split(dataset_cls, train):
    """Load a torchvision MNIST-like dataset as float images in [0, 1], shape (N, 1, 28, 28), and int64 labels."""
    try:
        ds = dataset_cls(DATA_DIR, train=train, download=True)
    except Exception as exc:
        raise RuntimeError(f"Could not load {dataset_cls.__name__} into {DATA_DIR}: {exc}. "
                           "The notebook needs an internet connection the first time it is run "
                           "(or copy the raw dataset files into ../data manually).") from exc
    return ds.data.unsqueeze(1).float().div(255.0), ds.targets.clone().long()

X_train_full, y_train_full = load_split(datasets.MNIST, train=True)
X_test_full, y_test_full = load_split(datasets.MNIST, train=False)

N_TRAIN = 6_000 if FAST_RUN else 55_000
N_VAL = 2_000 if FAST_RUN else 5_000
N_TEST = 2_000 if FAST_RUN else 10_000
EPOCHS = 1 if FAST_RUN else 4

perm = torch.randperm(len(X_train_full), generator=torch.Generator().manual_seed(SEED))
train_idx, val_idx = perm[:N_TRAIN], perm[N_TRAIN:N_TRAIN + N_VAL]
X_train, y_train = X_train_full[train_idx], y_train_full[train_idx]
X_val, y_val = X_train_full[val_idx], y_train_full[val_idx]
X_test, y_test = X_test_full[:N_TEST], y_test_full[:N_TEST]
train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=128, shuffle=True,
                          num_workers=0, pin_memory=(device.type == "cuda"))
print(f"train {tuple(X_train.shape)} | val {tuple(X_val.shape)} | test {tuple(X_test.shape)} | "
      f"pixel range [{X_train.min():.0f}, {X_train.max():.0f}]")

MNIST_MEAN, MNIST_STD = 0.1307, 0.3081

class SmallCNN(nn.Module):
    """Three conv stages and an MLP head with dropout. Takes images in [0, 1]; normalization is inside forward."""
    def __init__(self, dropout=0.3):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(),                          # [0-1]  16 x 28 x 28
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),        # [2-4]  32 x 14 x 14
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),        # [5-7]  64 x 7 x 7
        )
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Linear(64 * 7 * 7, 128), nn.ReLU(), nn.Dropout(dropout), nn.Linear(128, 10))

    def forward(self, x):
        return self.classifier(self.features((x - MNIST_MEAN) / MNIST_STD))

def train_one_epoch(model, loader, opt, attack=None, noise_std=0.0):
    """One epoch of cross-entropy training. `attack(model, x, y) -> x_adv` enables adversarial training,
    `noise_std > 0` enables Gaussian-noise augmentation. Returns (mean loss, accuracy) on the training batches."""
    model.train()
    total_loss = total_correct = n = 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        if noise_std > 0:
            xb = (xb + noise_std * torch.randn_like(xb)).clamp(0.0, 1.0)
        if attack is not None:
            model.eval()                     # craft the adversarial batch with a deterministic network
            xb = attack(model, xb, yb)
            model.train()
        logits = model(xb)
        loss = F.cross_entropy(logits, yb)
        opt.zero_grad(); loss.backward(); opt.step()
        total_loss += loss.item() * len(xb)
        total_correct += (logits.argmax(1) == yb).sum().item()
        n += len(xb)
    return total_loss / n, total_correct / n

@torch.no_grad()
def predict_logits(model, X, batch_size=512):
    """Logits of a whole tensor of images, computed in batches and returned on the CPU."""
    model.eval()
    return torch.cat([model(X[i:i + batch_size].to(device)).cpu() for i in range(0, len(X), batch_size)])

def evaluate(model, X, y):
    """(mean cross-entropy, accuracy) of a model on a tensor dataset."""
    logits = predict_logits(model, X)
    return F.cross_entropy(logits, y).item(), (logits.argmax(1) == y).float().mean().item()

def fit(model, loader, epochs, lr=1e-3, attack=None, noise_std=0.0, name="model", seed=SEED):
    """Train with Adam, printing one line per epoch; returns the model."""
    torch.manual_seed(seed)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    for ep in range(1, epochs + 1):
        t0 = time.time()
        tl, ta = train_one_epoch(model, loader, opt, attack=attack, noise_std=noise_std)
        vl, va = evaluate(model, X_val, y_val)
        print(f"{name} | epoch {ep}/{epochs} | train loss {tl:.3f} acc {ta:.3f} | val loss {vl:.3f} acc {va:.3f} | {time.time() - t0:.1f}s")
    return model

torch.manual_seed(SEED)
model = fit(SmallCNN().to(device), train_loader, EPOCHS, name="base CNN")
test_loss, test_acc = evaluate(model, X_test, y_test)
print(f"base CNN: test loss {test_loss:.3f}, test accuracy {test_acc:.4f}")

### 2.2 What an attribution method promises

An **attribution** (or **feature attribution**) method assigns to each input feature, here each pixel, a number that says how much that pixel contributed to a particular output, here the logit or probability of a class. The output is a heat-map of the same size as the image. Different methods answer subtly different questions ("what would change the output most if perturbed?", "what explains the output relative to a baseline?", "which region does the last layer attend to?"), and it is precisely because they differ that comparing them is informative. We implement five, each on a single image `x` of shape `(1, 1, 28, 28)` for a class index `target`, returning a `(28, 28)` NumPy array (Integrated Gradients additionally returns the sum of its map, for the completeness check below).

**Gradient saliency** (Simonyan et al., 2014) is the first-order answer. The class score $F_c(x)$ is approximated near $x$ by its Taylor expansion, $F_c(x + \delta) \approx F_c(x) + \delta^\top \nabla_x F_c(x)$, so the magnitude $|\partial F_c / \partial x_i|$ says how sensitive the score is to pixel $i$. It costs one backward pass. Its weakness is that the gradient of a piecewise-linear ReLU network is a *local* quantity that changes abruptly between adjacent linear regions, so the maps are noisy and speckled.

**SmoothGrad** (Smilkov et al., 2017) attacks exactly that problem: it averages the gradient over $n$ copies of the image with Gaussian noise added, $\frac{1}{n}\sum_k \nabla_x F_c(x + \epsilon_k)$, $\epsilon_k \sim \mathcal{N}(0, \sigma^2 I)$. Averaging over nearby linear regions removes the high-frequency speckle and leaves the structure that is consistent across the neighbourhood. A noise level of 10–20% of the pixel range and a few dozen copies is the usual recipe (the original paper found little further improvement beyond about 50 samples; we use 32).

**Integrated Gradients** (Sundararajan et al., 2017) is derived from axioms rather than from a Taylor expansion. Fix a **baseline** $x'$ that represents "absence of signal" (a black image for MNIST). Move along the straight path from $x'$ to $x$ and integrate the gradient along it:

$$
\mathrm{IG}_i(x) = (x_i - x'_i) \int_0^1 \frac{\partial F_c\big( x' + \alpha (x - x') \big)}{\partial x_i}\, d\alpha .
$$

The defining property is **completeness**. By the fundamental theorem of calculus applied to the scalar function $g(\alpha) = F_c(x' + \alpha (x - x'))$, whose derivative by the chain rule is $g'(\alpha) = \sum_i (x_i - x'_i)\, \partial_i F_c(x' + \alpha(x - x'))$, we get

$$
\sum_i \mathrm{IG}_i(x) = \int_0^1 g'(\alpha)\, d\alpha = g(1) - g(0) = F_c(x) - F_c(x') .
$$

The attributions sum exactly to the difference between the score of the image and the score of the baseline, so they have units (logits) and can be compared across images, and the method satisfies *sensitivity* (a feature that changes the output receives non-zero attribution) and *implementation invariance* (two networks computing the same function get the same attributions). Plain gradients satisfy the second but violate the first: a ReLU that is saturated at $x$ has zero gradient even when the feature clearly matters, which is the same locality problem that produces the speckle. Other popular methods that modify the backward pass (DeepLIFT, layer-wise relevance propagation) satisfy sensitivity but can violate implementation invariance. In practice the integral is a Riemann sum over $m \approx 50$ points, and the completeness identity is a built-in unit test: if the sum of the map is far from $F_c(x) - F_c(x')$, increase $m$.

**Occlusion** (Zeiler and Fergus, 2014) is model-agnostic and needs no gradients: slide a patch over the image, replace the pixels under it with the baseline value, and record how much the class probability drops. Regions whose occlusion hurts the prediction are important. It is slow (one forward pass per position, though the positions can be batched) and coarse, but it is the easiest method to explain and it directly measures a counterfactual.

**Grad-CAM** (Selvaraju et al., 2017), introduced with hooks in Notebook 07 · Transfer Learning and Modern Vision, works at the level of a convolutional feature map $A^k$ rather than pixels. It weights each channel by the spatially averaged gradient of the class score, $\alpha_k = \frac{1}{Z}\sum_{u,v} \partial F_c / \partial A^k_{uv}$, forms $\mathrm{ReLU}\big(\sum_k \alpha_k A^k\big)$, and upsamples the result to the input size. The ReLU keeps only regions that *increase* the class score. Because it operates on a low-resolution map (here $14 \times 14$) it is smooth and class-discriminative, but it cannot resolve fine detail; and because it depends on the chosen layer, it explains what that layer sees rather than what the pixels do.

In [ ]:
def saliency(model, x, target):
    """|dF_c/dx| for a single image x of shape (1, 1, H, W)."""
    model.eval()
    x = x.clone().requires_grad_(True)
    score = model(x)[0, target]
    grad, = torch.autograd.grad(score, x)
    return grad.abs().squeeze().cpu().numpy()

def smoothgrad(model, x, target, n_samples=32, sigma=0.15):
    """Average gradient over n noisy copies of x (noise std sigma in pixel units), then absolute value."""
    model.eval()
    xs = (x + sigma * torch.randn(n_samples, *x.shape[1:], device=x.device)).requires_grad_(True)
    score = model(xs)[:, target].sum()
    grad, = torch.autograd.grad(score, xs)
    return grad.mean(0).abs().squeeze().cpu().numpy()

def integrated_gradients(model, x, target, baseline=None, steps=50):
    """Integrated Gradients with a Riemann (midpoint) sum along the straight path from the baseline to x.
    Returns the attribution map and its sum (for the completeness check)."""
    model.eval()
    baseline = torch.zeros_like(x) if baseline is None else baseline
    alphas = ((torch.arange(steps, device=x.device) + 0.5) / steps).view(-1, 1, 1, 1)
    path = (baseline + alphas * (x - baseline)).requires_grad_(True)          # (steps, 1, H, W)
    score = model(path)[:, target].sum()
    grads, = torch.autograd.grad(score, path)
    ig = (x - baseline) * grads.mean(0, keepdim=True)
    return ig.squeeze().detach().cpu().numpy(), ig.sum().item()

@torch.no_grad()
def occlusion(model, x, target, patch=6, stride=2, fill=0.0):
    """Drop in the class probability when a patch x patch square is replaced by `fill`, averaged per pixel."""
    model.eval()
    H, W = x.shape[-2:]
    base_prob = F.softmax(model(x), 1)[0, target].item()
    positions = [(r, c) for r in range(0, H - patch + 1, stride) for c in range(0, W - patch + 1, stride)]
    xs = x.repeat(len(positions), 1, 1, 1)
    for i, (r, c) in enumerate(positions):
        xs[i, :, r:r + patch, c:c + patch] = fill
    probs = F.softmax(model(xs), 1)[:, target]
    heat = torch.zeros(H, W, device=x.device)
    count = torch.zeros(H, W, device=x.device)
    for i, (r, c) in enumerate(positions):
        heat[r:r + patch, c:c + patch] += base_prob - probs[i]
        count[r:r + patch, c:c + patch] += 1
    return (heat / count.clamp(min=1)).cpu().numpy()

def grad_cam(model, x, target, layer=None):
    """Grad-CAM on the output of `layer` (default: model.features[6], the ReLU after the last convolution,
    64 channels at 14 x 14), upsampled to the input size and scaled to [0, 1]."""
    model.eval()
    layer = model.features[6] if layer is None else layer
    store = {}
    handle = layer.register_forward_hook(lambda m, i, o: store.__setitem__("act", o))
    score = model(x)[0, target]
    handle.remove()
    grads, = torch.autograd.grad(score, store["act"])
    weights = grads.mean(dim=(2, 3), keepdim=True)                          # alpha_k: (1, C, 1, 1)
    cam = F.relu((weights * store["act"]).sum(1, keepdim=True))            # (1, 1, h, w)
    cam = F.interpolate(cam, size=x.shape[-2:], mode="bilinear", align_corners=False)
    return (cam / (cam.max() + 1e-8)).squeeze().detach().cpu().numpy()

# Completeness check for Integrated Gradients on one test image.
x0 = X_test[:1].to(device)
c0 = int(y_test[0])
with torch.no_grad():
    f_x, f_base = model(x0)[0, c0].item(), model(torch.zeros_like(x0))[0, c0].item()
for steps in [5, 20, 50, 200]:
    _, ig_sum = integrated_gradients(model, x0, c0, steps=steps)
    print(f"IG with {steps:3d} steps: sum of attributions = {ig_sum:8.3f}   vs   F(x) - F(baseline) = {f_x - f_base:8.3f}")

The completeness identity holds to within the Riemann-sum error, which shrinks as the number of steps grows; 50 steps is a good default for a network this size. Now we compute all five maps for three test digits and display them side by side. Attribution maps are shown with a red colour scale next to the digit; for saliency and SmoothGrad the map is the absolute gradient, for Integrated Gradients we show the signed attribution (red for pixels that push towards the class, blue for pixels that push against it), for occlusion the probability drop, and for Grad-CAM the class activation map.

In [ ]:
def attribution_maps(model, x, target):
    return {"saliency": saliency(model, x, target),
            "SmoothGrad": smoothgrad(model, x, target),
            "Integrated Gradients": integrated_gradients(model, x, target)[0],
            "occlusion": occlusion(model, x, target),
            "Grad-CAM": grad_cam(model, x, target)}

def show_attribution_grid(model, indices, title):
    methods = ["saliency", "SmoothGrad", "Integrated Gradients", "occlusion", "Grad-CAM"]
    fig, axes = plt.subplots(len(indices), len(methods) + 1, figsize=(2.0 * (len(methods) + 1), 2.1 * len(indices)))
    for row, idx in enumerate(indices):
        x = X_test[idx:idx + 1].to(device)
        target = int(y_test[idx])
        with torch.no_grad():
            prob = F.softmax(model(x), 1)[0, target].item()
        maps = attribution_maps(model, x, target)
        axes[row, 0].imshow(x.squeeze().cpu(), cmap="gray"); axes[row, 0].set_title(f"label {target}, p = {prob:.2f}", fontsize=9)
        for col, name in enumerate(methods, start=1):
            m = maps[name]
            if name == "Integrated Gradients":
                v = np.abs(m).max() + 1e-8
                axes[row, col].imshow(m, cmap="bwr", vmin=-v, vmax=v)
            else:
                axes[row, col].imshow(m, cmap="Reds")
            if row == 0:
                axes[row, col].set_title(name, fontsize=9)
        for ax in axes[row]:
            ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
    fig.suptitle(title); plt.tight_layout(); plt.show()

example_indices = [int((y_test == d).nonzero()[0]) for d in (3, 7, 8)]
show_attribution_grid(model, example_indices, "Attribution maps of the trained CNN")

Several patterns are typical and worth pointing out. Saliency is speckled and highlights pixels both on and next to the strokes, because the gradient of a ReLU network is dominated by which units happen to be active at exactly this input. SmoothGrad removes the speckle and concentrates on the strokes. Integrated Gradients is the only map with signed values in units of logits; the blue regions are informative: for the "3", pixels along the left side often carry *negative* attribution, because ink there would turn the 3 into an 8, and their emptiness supports the prediction. Occlusion is coarse but easy to read (the drop is largest where a patch would remove a distinctive part of the stroke, such as the middle bar of the 3 or the junctions of the 8), and it is the only map that says nothing about pixels outside the digit, because occluding an already-black region with black changes nothing. Grad-CAM is smooth and, at this resolution, mostly locates the digit as a whole.

> **Common pitfall.** Attribution maps are visualizations of a *class score*, and the choice of score matters: attributing the softmax probability instead of the logit gives maps that also reflect the other classes (a pixel can raise the probability of "3" by lowering the logit of "8"). All our gradient-based functions attribute the logit, which is the more common convention (occlusion, by design, measures the drop in probability); keep this in mind when comparing with papers or libraries.

### 2.3 Sanity checks: does the explanation depend on the model?

An attribution method can produce a beautiful map that has nothing to do with the model. Adebayo et al. (2018) proposed a simple test: randomize the model's weights and recompute the map. A method that explains the *model* must produce a different map for a random model; a method whose output looks the same is acting as an edge detector on the input. Several popular methods (guided backpropagation among them, see Exercise 1) fail this test. We run the **model randomization test** on our five methods by comparing each map of the trained model with the map of a freshly initialized network, using the Spearman rank correlation between the two maps (rank correlation is appropriate because the maps have different scales and we care about whether the same pixels are ranked as important).

In [ ]:
def rank_correlation(a, b):
    """Spearman rank correlation between two maps (absolute values, flattened; tied values get their average rank)."""
    ra, rb = rankdata(np.abs(a).ravel()), rankdata(np.abs(b).ravel())
    return float(np.corrcoef(ra, rb)[0, 1])

torch.manual_seed(SEED + 1)
random_model = SmallCNN().to(device).eval()          # same architecture, untrained weights

corrs = {name: [] for name in ["saliency", "SmoothGrad", "Integrated Gradients", "occlusion", "Grad-CAM"]}
for idx in example_indices:
    x = X_test[idx:idx + 1].to(device); target = int(y_test[idx])
    maps_trained = attribution_maps(model, x, target)
    maps_random = attribution_maps(random_model, x, target)      # Grad-CAM hooks the random model's own last ReLU
    for name in corrs:
        corrs[name].append(rank_correlation(maps_trained[name], maps_random[name]))

print("rank correlation between the maps of the trained and of a randomly initialized model (lower = more model-dependent):")
for name, vals in corrs.items():
    print(f"   {name:22s}: {np.mean(vals):.3f}")

# Visual version of the test for the first example image.
idx = example_indices[0]
x = X_test[idx:idx + 1].to(device); target = int(y_test[idx])
fig, axes = plt.subplots(2, 5, figsize=(10, 4.2))
for row, (m, label) in enumerate([(model, "trained"), (random_model, "random weights")]):
    maps = attribution_maps(m, x, target)
    for col, name in enumerate(corrs):
        axes[row, col].imshow(np.abs(maps[name]), cmap="Reds"); axes[row, col].set_xticks([]); axes[row, col].set_yticks([]); axes[row, col].grid(False)
        axes[row, col].set_title(f"{name}\n({label})", fontsize=8)
plt.suptitle("Model randomization test"); plt.tight_layout(); plt.show()

The methods split into two groups, and the split agrees with what Adebayo et al. reported for the methods they tested (occlusion was not among them, but the same reasoning applies). Saliency, SmoothGrad and Grad-CAM change substantially when the weights are randomized: their correlations are a few tenths at most (Grad-CAM's is close to zero, since a random network's class activation map is essentially noise), with a residual similarity for the gradient maps because a random ReLU network still has larger gradients where the image has edges. Integrated Gradients and occlusion, on the other hand, stay highly correlated with their trained-model versions, sometimes above 0.9. Both are **input-dominated**: Integrated Gradients multiplies the averaged gradient by $(x - x')$, which with a black baseline is the digit itself, so the *ranking* of pixels is largely fixed by where the ink is, whatever the gradients say; and occluding a black region with black changes nothing, so the occlusion map of *any* model can only be non-zero on the strokes. (The completeness property still holds for the random model; the sanity check tests something else, namely whether the *pattern* depends on the weights.) This does not make the two methods useless, since the magnitudes and signs of their attributions do depend on the model, but it means that the visual impression of "the model looks at the digit" would appear for a model that has learned nothing. Whenever you adopt an attribution method, run this test on your own model; it is cheap, and it is the difference between an explanation and a decoration.

Sanity checks of this kind test *necessary* conditions only. Even a method that passes can be misread: a map shows which pixels the model's score is sensitive to, not whether that sensitivity is "reasonable", and interpreting it requires domain knowledge. The huskies-versus-wolves case was caught because a human looked at the map and saw snow; the map did not say "shortcut".

## 3. Feature visualization and probing representations

### 3.1 Activation maximization

Attribution explains one prediction. **Feature visualization** asks what a unit responds to in general, by synthesizing an input that maximizes its activation (Erhan et al., 2009; Olah et al., 2017). Starting from noise, we perform gradient *ascent* on the input to maximize the mean activation of a chosen channel of a convolutional layer, or the logit of a class. Unconstrained, this produces high-frequency adversarial-looking patterns rather than anything a human can read (a first hint of Section 4), so regularization is essential. We use three standard pieces: the image is parametrized as $x = \sigma(w)$ so that it stays in $[0, 1]$; a **total-variation** penalty $\sum |x_{i+1,j} - x_{i,j}| + |x_{i,j+1} - x_{i,j}|$ discourages high-frequency noise; and **jitter** (a random shift of a few pixels before each forward pass) forces the pattern to be robust to small translations, which is what natural features are. A small $L_2$ penalty keeps the image from saturating.

The function below optimizes a whole batch of images at once, one per objective, so that visualizing eight channels or all ten classes takes a single run.

In [ ]:
def total_variation(img):
    """Per-image total variation of a (B, 1, H, W) tensor, averaged over pixels."""
    dh = (img[:, :, 1:, :] - img[:, :, :-1, :]).abs().mean(dim=(1, 2, 3))
    dw = (img[:, :, :, 1:] - img[:, :, :, :-1]).abs().mean(dim=(1, 2, 3))
    return dh + dw

def maximize_activation(model, objective, n_images, steps=200, lr=0.05, tv_weight=3.0, l2_weight=0.3, jitter=2, seed=SEED):
    """Gradient ascent on n_images inputs so that objective(images) -> (n_images,) is maximized,
    with a sigmoid parametrization, total-variation and L2 penalties, and random jitter."""
    model.eval()
    g = torch.Generator(device="cpu").manual_seed(seed)
    w = (0.1 * torch.randn(n_images, 1, 28, 28, generator=g)).to(device).requires_grad_(True)
    opt = torch.optim.Adam([w], lr=lr)
    rng = np.random.default_rng(seed)
    for step in range(steps):
        img = torch.sigmoid(w)
        dx, dy = (int(v) for v in rng.integers(-jitter, jitter + 1, size=2))
        shifted = torch.roll(img, shifts=(dx, dy), dims=(2, 3))
        loss = (-objective(shifted) + tv_weight * total_variation(img) + l2_weight * img.pow(2).mean(dim=(1, 2, 3))).sum()
        grad, = torch.autograd.grad(loss, w)
        w.grad = grad
        opt.step()
    with torch.no_grad():
        final = torch.sigmoid(w)
        return final, objective(final).detach()

normalize = lambda x: (x - MNIST_MEAN) / MNIST_STD
channels = list(range(8))
def channel_objective(imgs):
    acts = model.features[:7](normalize(imgs))                   # (B, 64, 14, 14), after the ReLU of conv3
    return torch.stack([acts[i, k].mean() for i, k in enumerate(channels)])
def class_objective(imgs):
    return model(imgs)[torch.arange(10, device=imgs.device), torch.arange(10, device=imgs.device)]

t0 = time.time()
chan_imgs, chan_vals = maximize_activation(model, channel_objective, len(channels))
class_imgs, class_vals = maximize_activation(model, class_objective, 10)
print(f"optimized 8 channel images and 10 class images in {time.time() - t0:.1f}s")

with torch.no_grad():
    test_logits = predict_logits(model, X_test)
real_class_logits = torch.stack([test_logits[y_test == c, c].mean() for c in range(10)])
print("class logit of the synthesized image vs mean logit of real test images of that class:")
for c in range(10):
    print(f"   class {c}: synthesized {class_vals[c].item():6.1f}   real {real_class_logits[c].item():6.1f}")

fig, axes = plt.subplots(2, 10, figsize=(12, 3))
for i in range(10):
    if i < len(channels):
        axes[0, i].imshow(chan_imgs[i, 0].cpu(), cmap="gray"); axes[0, i].set_title(f"conv3 ch {channels[i]}", fontsize=8)
    else:
        axes[0, i].axis("off")
    axes[1, i].imshow(class_imgs[i, 0].cpu(), cmap="gray"); axes[1, i].set_title(f"class {i}", fontsize=8)
for ax in axes.ravel():
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
plt.suptitle("Activation maximization: conv3 channels (top) and class logits (bottom)"); plt.tight_layout(); plt.show()

The synthesized images drive their logits well above anything a real digit achieves, which is the first lesson: a network's preferred input is not a typical input. The class images nevertheless carry recognizable fragments of digit structure (loops for 0, 6, 8, 9; a bar for 1; the closed lower part of a 3), especially in the full run, while the channel images show more local motifs (oriented strokes, curves, stroke endings) that the last convolutional layer detects. In larger networks trained on natural images this technique produces the well-known "dream" visualizations of eyes, fur and text fragments, and it is the basis of circuit-level analyses of vision models. Its limitation is that a unit's preferred stimulus is not the same as its role: many channels respond to several unrelated patterns (**polysemanticity**), and the synthesized image shows only one.

> **Try it.** Set `tv_weight=0.0` and `jitter=0`. The optimized images become high-frequency noise that nonetheless produces enormous logits. That is exactly an adversarial example built from scratch, and Section 4 explains why it exists.

### 3.2 Linear probes: how decodable is the label at each depth?

A complementary way to understand a representation is to ask what can be *read out* of it. A **linear probe** (Alain and Bengio, 2017) is a logistic-regression classifier trained on the frozen activations of one layer; its accuracy measures how linearly separable the classes are at that depth. The raw pixels are the baseline (a linear classifier on MNIST pixels reaches about 92%), and as we go deeper the accuracy should increase, reflecting the gradual untangling of the classes that the network performs. Probes are also a standard tool in NLP, where they are used to ask whether a language model's hidden states encode part-of-speech, syntax trees or factual attributes.

We extract features at five points (pixels; after each of the three convolutional stages, spatially average-pooled to $4 \times 4$ to keep the dimensionality comparable; and after the hidden fully-connected layer), standardize them, and train a linear classifier on each with a few hundred full-batch steps.

In [ ]:
PROBE_POINTS = ["pixels", "conv1", "conv2", "conv3", "fc1"]
CONV_SLICES = {"conv1": 2, "conv2": 5, "conv3": 8}      # model.features[:k] ends after the ReLU (and pool) of that stage

@torch.no_grad()
def extract_features(model, X, point, batch_size=512):
    """Frozen activations at a probe point ('pixels', 'conv1', 'conv2', 'conv3' or 'fc1');
    conv maps are average-pooled to 4x4 and flattened."""
    model.eval()
    out = []
    for i in range(0, len(X), batch_size):
        xb = X[i:i + batch_size].to(device)
        if point == "pixels":
            f = xb.flatten(1)
        elif point == "fc1":
            f = model.classifier[:3](model.features(normalize(xb)))
        else:
            f = F.adaptive_avg_pool2d(model.features[:CONV_SLICES[point]](normalize(xb)), 4).flatten(1)
        out.append(f.cpu())
    return torch.cat(out)

def train_probe(F_train, y_train_, F_test, y_test_, steps=300, lr=1e-2, weight_decay=1e-4):
    """Logistic regression on standardized features; returns test accuracy."""
    mu, sd = F_train.mean(0, keepdim=True), F_train.std(0, keepdim=True) + 1e-6
    Ftr, Fte = ((F_train - mu) / sd).to(device), ((F_test - mu) / sd).to(device)
    ytr, yte = y_train_.to(device), y_test_.to(device)
    torch.manual_seed(SEED)
    probe = nn.Linear(Ftr.shape[1], 10).to(device)
    opt = torch.optim.Adam(probe.parameters(), lr=lr, weight_decay=weight_decay)
    for _ in range(steps):
        loss = F.cross_entropy(probe(Ftr), ytr)
        opt.zero_grad(); loss.backward(); opt.step()
    with torch.no_grad():
        return (probe(Fte).argmax(1) == yte).float().mean().item()

N_PROBE = 4_000 if FAST_RUN else 20_000
probe_acc = {}
for point in PROBE_POINTS:
    t0 = time.time()
    Ftr = extract_features(model, X_train[:N_PROBE], point)
    Fte = extract_features(model, X_test, point)
    probe_acc[point] = train_probe(Ftr, y_train[:N_PROBE], Fte, y_test)
    print(f"probe on {point:7s}: {Ftr.shape[1]:5d} features -> test accuracy {probe_acc[point]:.4f}   ({time.time() - t0:.1f}s)")

plt.figure()
plt.bar(list(probe_acc.keys()), list(probe_acc.values()), color="steelblue")
plt.axhline(test_acc, color="k", ls="--", label=f"full CNN ({test_acc:.3f})")
plt.ylim(min(0.8, min(probe_acc.values()) - 0.02), 1.0); plt.ylabel("linear probe test accuracy"); plt.title("Linear decodability of the label by depth"); plt.legend()
plt.tight_layout(); plt.show()

With the fully trained model, linear decodability rises with depth: a linear classifier on raw pixels reaches about 92%, the pooled convolutional features do progressively better, and the last hidden layer is by construction the best (the network's own output layer *is* a linear probe on it, and reaches the model's 99%). Two caveats are visible in the numbers, especially in the `FAST_RUN` version. First, the early convolutional features, pooled to $4 \times 4$, can be *worse* than the pixels, because pooling discards spatial detail that the pixel classifier uses: a probe measures the information accessible to a *linear* reader of the particular features we chose, not the information present in the layer. Second, a probe measures the representation *as it is*: after a single epoch on 6,000 images the network's own `fc1` features are still crude, and a well-trained linear probe on pixels or on the mid-level features can beat them. Probes are therefore most informative for comparing layers of a converged model, and their absolute values should always be read next to the model's own accuracy (the dashed line).

### 3.3 Concept-based explanations

Attributions live in pixel space and probes need labels. **Testing with Concept Activation Vectors** (TCAV, Kim et al., 2018) works with human concepts instead. The user supplies a small set of example images of a concept ("striped", "has wheels") and a set of random images; a linear classifier separating the two in the activation space of some layer gives a direction $v_C$, the concept activation vector. The directional derivative of the class score along $v_C$, $\partial F_k / \partial v_C$, measures whether moving the activation towards the concept increases the class score, and the TCAV score of concept $C$ for class $k$ is the fraction of class-$k$ images for which this derivative is positive. Statistical significance is assessed by repeating the procedure with several random sets. The method can answer questions like "does the model use the concept *snow* to predict *wolf*?" without any pixel-level map, and it is the natural tool for auditing the shortcut-learning failures of Section 1. Its requirements are a layer at which the concept is linearly encoded and enough example images to define it, which is why it is most used in domains (medical imaging, natural images) where such concept sets can be curated.

Notebook 16 · Interpretability, Robustness and Uncertainty — part 2 (adversarial examples, distribution shift and OOD detection)

## 4. Adversarial examples and adversarial training

### 4.1 Why adversarial examples exist: the linearity argument

An **adversarial example** is an input $x + \eta$ that differs from a correctly classified input $x$ by a perturbation $\eta$ too small for a person to notice, yet is classified differently, usually with high confidence. When Szegedy et al. (2014) found them, the natural guess was that they exploit the extreme non-linearity of deep networks. Goodfellow, Shlens and Szegedy (2015) argued the opposite: adversarial examples are a consequence of networks being *too linear* in high dimensions.

Consider a linear score $s(x) = w^\top x$ with $x \in \mathbb{R}^n$, and a perturbation bounded in the max-norm, $\|\eta\|_\infty \le \epsilon$ (no pixel changes by more than $\epsilon$). The change in the score is $w^\top \eta$, which is maximized by $\eta = \epsilon\, \mathrm{sign}(w)$ and then equals $\epsilon \|w\|_1 = \epsilon\, n\, \overline{|w|}$, where $\overline{|w|}$ is the average magnitude of the weights. The perturbation of each pixel is tiny, but there are $n = 784$ of them for MNIST and $150{,}528$ for a $224 \times 224$ colour image, and their effects on the score *add up*. A change that is invisible per pixel becomes enormous in the output, and the more input dimensions there are, the worse it gets. Deep networks with ReLUs are piecewise linear, and their pieces are large enough for the argument to apply locally; it also explains the noise images of Section 3.1 that produced huge logits.

### 4.2 The Fast Gradient Sign Method

The argument suggests the attack. Let $L(x, y)$ be the loss of the model on input $x$ with true label $y$. We want the perturbation within the $\epsilon$-ball that *increases the loss the most*. Linearizing the loss around $x$, $L(x + \eta, y) \approx L(x, y) + \eta^\top \nabla_x L(x, y)$, and maximizing the linear term under $\|\eta\|_\infty \le \epsilon$ gives, by the same computation as above,

$$
\eta^{\text{FGSM}} = \epsilon\, \mathrm{sign}\big( \nabla_x L(x, y) \big), \qquad x^{\text{adv}} = \mathrm{clip}_{[0,1]}\big( x + \eta^{\text{FGSM}} \big).
$$

This is the **Fast Gradient Sign Method**: one backward pass with respect to the *input*, a sign, a step. Note the contrast with training: there we take a gradient step on the parameters to *decrease* the loss; here we take a step on the input to *increase* it. The clip keeps the result a valid image.

### 4.3 Projected Gradient Descent

FGSM trusts the linear approximation for the whole step. **Projected gradient descent** (PGD, Madry et al., 2018) instead takes $K$ smaller steps of size $\alpha$, projecting back onto the $\epsilon$-ball (and the valid pixel range) after each,

$$
x_{k+1} = \Pi_{\mathcal{B}_\epsilon(x) \cap [0,1]^n}\Big( x_k + \alpha\, \mathrm{sign}\big( \nabla_x L(x_k, y) \big) \Big), \qquad x_0 = x + u,\; u \sim \mathcal{U}(-\epsilon, \epsilon)^n .
$$

For the max-norm ball the projection is a per-pixel clamp to $[x_i - \epsilon, x_i + \epsilon]$. The random start $u$ avoids the flat region around $x$ where the gradient can be uninformative, and several restarts make the attack stronger still. PGD is the standard *first-order* adversary: a model that resists it with a given $\epsilon$ is considered empirically robust at that radius, and it is also the attack used *inside* adversarial training (Section 4.6).

Both attacks have a **targeted** variant: replace the true label by a chosen target $t$ and take steps that *decrease* the loss $L(x, t)$, pushing the input towards the target class. We implement both attacks with a `targeted` flag and measure accuracy as a function of $\epsilon$ on a subset of the test set.

In [ ]:
def fgsm(model, x, y, eps, targeted=False):
    """Fast Gradient Sign Method: one step of size eps along the sign of the input gradient of the loss.
    With targeted=True, y is the target class and the step decreases the loss towards it."""
    x_adv = x.clone().requires_grad_(True)
    loss = F.cross_entropy(model(x_adv), y)
    grad, = torch.autograd.grad(loss, x_adv)
    direction = -grad.sign() if targeted else grad.sign()
    return (x_adv + eps * direction).clamp(0.0, 1.0).detach()

def pgd(model, x, y, eps, steps=10, alpha=None, targeted=False, random_start=True):
    """Projected gradient descent in the L_inf ball of radius eps around x (Madry et al., 2018)."""
    alpha = 2.5 * eps / steps if alpha is None else alpha
    x_adv = x.clone()
    if random_start:
        x_adv = (x_adv + torch.empty_like(x_adv).uniform_(-eps, eps)).clamp(0.0, 1.0)
    for _ in range(steps):
        x_adv.requires_grad_(True)
        loss = F.cross_entropy(model(x_adv), y)
        grad, = torch.autograd.grad(loss, x_adv)
        with torch.no_grad():
            x_adv = x_adv + (-alpha if targeted else alpha) * grad.sign()
            x_adv = torch.min(torch.max(x_adv, x - eps), x + eps).clamp(0.0, 1.0)   # project onto the ball and [0, 1]
    return x_adv.detach()

def accuracy_under_attack(model, X, y, attack, eps_values, batch_size=250, **attack_kwargs):
    """Accuracy of `model` on adversarial versions of (X, y) for every eps in eps_values (eps = 0 is clean accuracy)."""
    model.eval()
    accs = []
    for eps in eps_values:
        correct = 0
        for i in range(0, len(X), batch_size):
            xb, yb = X[i:i + batch_size].to(device), y[i:i + batch_size].to(device)
            x_adv = xb if eps == 0 else attack(model, xb, yb, eps, **attack_kwargs)
            with torch.no_grad():
                correct += (model(x_adv).argmax(1) == yb).sum().item()
        accs.append(correct / len(X))
    return accs

N_ADV = 500 if FAST_RUN else 2_000
X_adv_eval, y_adv_eval = X_test[:N_ADV], y_test[:N_ADV]
EPS_VALUES = [0.0, 0.025, 0.05, 0.1, 0.15, 0.2, 0.3]

t0 = time.time()
acc_fgsm = accuracy_under_attack(model, X_adv_eval, y_adv_eval, fgsm, EPS_VALUES)
acc_pgd = accuracy_under_attack(model, X_adv_eval, y_adv_eval, pgd, EPS_VALUES, steps=10)
print(f"attacks on {N_ADV} test images took {time.time() - t0:.1f}s")
print(f"{'eps':>6s} {'FGSM acc':>10s} {'PGD-10 acc':>11s}")
for eps, a_f, a_p in zip(EPS_VALUES, acc_fgsm, acc_pgd):
    print(f"{eps:6.3f} {a_f:10.3f} {a_p:11.3f}")

plt.figure()
plt.plot(EPS_VALUES, acc_fgsm, "o-", label="FGSM")
plt.plot(EPS_VALUES, acc_pgd, "s-", label="PGD (10 steps)")
plt.xlabel("perturbation budget epsilon (L_inf, pixel units in [0, 1])"); plt.ylabel("accuracy")
plt.title("Accuracy of the standard CNN under attack"); plt.legend(); plt.tight_layout(); plt.show()

# What the perturbations look like.
x = X_test[:1].to(device); y = y_test[:1].to(device)
x_f, x_p = fgsm(model, x, y, 0.1), pgd(model, x, y, 0.1, steps=10)
with torch.no_grad():
    probs = [F.softmax(model(v), 1)[0] for v in (x, x_f, x_p)]
fig, axes = plt.subplots(1, 3, figsize=(7.5, 2.8))
for ax, img, p, name in zip(axes, (x, x_f, x_p), probs, ("clean", "FGSM eps=0.1", "PGD eps=0.1")):
    ax.imshow(img.squeeze().cpu(), cmap="gray", vmin=0, vmax=1); ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
    ax.set_title(f"{name}\npred {int(p.argmax())} (p = {p.max():.2f})", fontsize=9)
plt.tight_layout(); plt.show()

Accuracy collapses long before the perturbation becomes visible. At $\epsilon = 0.1$, a tenth of the pixel range, the ten-step PGD attack already flips a large fraction of the predictions, and at $\epsilon = 0.2$ it brings the standard CNN close to zero accuracy, while the images still read unmistakably as the same digits (MNIST is unusually forgiving here because its pixels are nearly binary; on natural images $\epsilon = 8/255 \approx 0.03$ suffices). PGD is consistently stronger than FGSM at the same budget, as the multi-step optimization finds perturbations that the single linear step misses; it is also the reason a model must be evaluated against PGD, not FGSM, before any robustness claim is made.

### 4.4 Targeted attacks

The targeted variant lets an attacker choose the outcome. We take a test image of a 7 and ask PGD to turn it into a 3. Reaching one *specific* wrong class is harder than reaching *any* wrong class, so we allow a larger budget, $\epsilon = 0.3$ with 100 steps, and then look at the perturbation itself.

In [ ]:
idx7 = int((y_test == 7).nonzero()[0])
x7 = X_test[idx7:idx7 + 1].to(device)
target_cls = torch.tensor([3], device=device)
EPS_TARGETED = 0.3
x7_adv = pgd(model, x7, target_cls, eps=EPS_TARGETED, steps=100, targeted=True)
with torch.no_grad():
    p_clean, p_adv = F.softmax(model(x7), 1)[0], F.softmax(model(x7_adv), 1)[0]
print(f"clean image     : predicted {int(p_clean.argmax())} with p = {p_clean.max():.3f}")
print(f"targeted PGD    : predicted {int(p_adv.argmax())} with p = {p_adv.max():.3f}   "
      f"(p(3) went from {p_clean[3]:.4f} to {p_adv[3]:.3f}; max |perturbation| = {(x7_adv - x7).abs().max():.3f})")
print("attack " + ("succeeded: the model now reads the 7 as a 3." if int(p_adv.argmax()) == 3 else
                   "did not reach the target within this budget; try a larger eps or more steps."))

delta = (x7_adv - x7).squeeze().cpu()
fig, axes = plt.subplots(1, 3, figsize=(7.5, 2.8))
axes[0].imshow(x7.squeeze().cpu(), cmap="gray", vmin=0, vmax=1); axes[0].set_title("original (7)", fontsize=9)
axes[1].imshow(delta, cmap="bwr", vmin=-EPS_TARGETED, vmax=EPS_TARGETED); axes[1].set_title("perturbation (red = added ink)", fontsize=9)
axes[2].imshow(x7_adv.squeeze().cpu(), cmap="gray", vmin=0, vmax=1); axes[2].set_title(f"adversarial -> {int(p_adv.argmax())}", fontsize=9)
for ax in axes:
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
plt.tight_layout(); plt.show()

The perturbation is not random noise: it typically adds ink where a 3 would have strokes (the curved right-hand side and the middle bar) and removes it along the 7's diagonal, which is what the gradient of the target-class loss encodes. With the budget used here the adversarial image usually still reads as a 7 to a person while the model prefers 3; with a larger budget the image would visibly turn into a 3, and with a smaller one the attack fails, which is itself informative: the distance to a *chosen* class is a property of the model's geometry that untargeted accuracy curves do not reveal. Targeted attacks are the relevant threat model whenever a specific wrong output is valuable to an attacker (a stop sign read as a speed limit, a malware sample read as benign).

### 4.5 Transferability

The attacks so far are **white-box**: they use the gradient of the attacked model. Surprisingly, adversarial examples crafted for one model often fool a *different* model trained on the same task, even with a different architecture. This **transferability** (Papernot et al., 2016) is what makes black-box attacks practical: an attacker trains a surrogate model, attacks it, and sends the results to the deployed system. The linearity argument explains it: different models trained on the same data learn similar linear-ish decision boundaries in similar directions.

We train a second model with a different architecture, a plain MLP, craft PGD examples on each model, and evaluate both models on both sets.

In [ ]:
class MLP(nn.Module):
    """Two-hidden-layer MLP on flattened pixels (a different architecture for the transfer experiment)."""
    def __init__(self, hidden=256):
        super().__init__()
        self.net = nn.Sequential(nn.Flatten(), nn.Linear(784, hidden), nn.ReLU(), nn.Linear(hidden, hidden), nn.ReLU(), nn.Linear(hidden, 10))
    def forward(self, x):
        return self.net((x - MNIST_MEAN) / MNIST_STD)

torch.manual_seed(SEED + 2)
mlp = fit(MLP().to(device), train_loader, EPOCHS, name="MLP", seed=SEED + 2)

def eval_on(model_, X_, y_):
    with torch.no_grad():
        model_.eval()
        return (model_(X_).argmax(1) == y_).float().mean().item()

xb, yb = X_adv_eval[:N_ADV].to(device), y_adv_eval[:N_ADV].to(device)
sources = {"CNN": model, "MLP": mlp}
print("accuracy on PGD examples (eps = 0.15, 10 steps); rows = model that crafted them, columns = model evaluated")
header = "crafted on / evaluated"
print(f"{header:24s} {'CNN':>8s} {'MLP':>8s}")
for src_name, src_model in sources.items():
    x_adv = pgd(src_model, xb, yb, eps=0.15, steps=10)
    accs = [eval_on(m, x_adv, yb) for m in sources.values()]
    print(f"{src_name:24s} {accs[0]:8.3f} {accs[1]:8.3f}")
print(f"{'clean':24s} {eval_on(model, xb, yb):8.3f} {eval_on(mlp, xb, yb):8.3f}")

The diagonal entries are the white-box results (near zero accuracy). The off-diagonal entries are the transfer results: examples crafted on the MLP substantially reduce the CNN's accuracy and vice versa, even though neither model's gradient was used against the other. Transfer is partial (the victim retains more accuracy than under a white-box attack) and asymmetric (attacks from a model with smoother, more "generic" gradients transfer better), but it is far from negligible, and it means that keeping a model's weights secret is not a defence.

### 4.6 Adversarial training and the robustness–accuracy trade-off

The most reliable defence known is also the most direct: train on adversarial examples. Madry et al. (2018) formulate robust training as a **min-max** problem,

$$
\min_\theta\; \mathbb{E}_{(x, y) \sim \mathcal{D}} \Big[ \max_{\|\eta\|_\infty \le \epsilon} L\big( f_\theta(x + \eta),\, y \big) \Big],
$$

where the inner maximization is solved approximately by PGD for every mini-batch and the outer minimization by SGD on the resulting worst-case inputs. The network is thereby trained to be accurate on the entire $\epsilon$-ball around every training image, not just at its centre. The cost is $K + 1$ forward-backward passes per batch instead of one, and, more fundamentally, a **trade-off**: robust models have lower clean accuracy than standard ones, because features that are predictive but non-robust (fine-grained, high-frequency patterns that a small perturbation can flip) must be given up (Tsipras et al., 2019). Robust models also have visibly different gradients, which we check at the end of this section.

Under `FAST_RUN` we train for one epoch on the subset with a 5-step PGD adversary at $\epsilon = 0.15$; the full run uses three epochs and seven steps. The training attack is weaker than the evaluation attack (10 steps), which is the correct direction: a model should be evaluated against a stronger adversary than it was trained on.

In [ ]:
ADV_EPS = 0.15
ADV_STEPS = 5 if FAST_RUN else 7
ADV_EPOCHS = 1 if FAST_RUN else 3

torch.manual_seed(SEED)
robust_model = fit(SmallCNN().to(device), train_loader, ADV_EPOCHS, name="PGD-trained CNN",
                   attack=lambda m, x_, y_: pgd(m, x_, y_, ADV_EPS, steps=ADV_STEPS))

t0 = time.time()
acc_pgd_robust = accuracy_under_attack(robust_model, X_adv_eval, y_adv_eval, pgd, EPS_VALUES, steps=10)
print(f"evaluation took {time.time() - t0:.1f}s")
print(f"{'eps':>6s} {'standard CNN':>13s} {'robust CNN':>11s}")
for eps, a_s, a_r in zip(EPS_VALUES, acc_pgd, acc_pgd_robust):
    print(f"{eps:6.3f} {a_s:13.3f} {a_r:11.3f}")

plt.figure()
plt.plot(EPS_VALUES, acc_pgd, "s-", label="standard training")
plt.plot(EPS_VALUES, acc_pgd_robust, "o-", label=f"PGD adversarial training (eps = {ADV_EPS})")
plt.axvline(ADV_EPS, color="gray", ls=":", label="training epsilon")
plt.xlabel("PGD budget epsilon"); plt.ylabel("accuracy"); plt.title("The robustness-accuracy trade-off"); plt.legend()
plt.tight_layout(); plt.show()

# Gradients of robust models look different: saliency maps of both models on the same digits.
fig, axes = plt.subplots(2, 3, figsize=(6, 4))
for col, idx in enumerate(example_indices):
    x = X_test[idx:idx + 1].to(device); t = int(y_test[idx])
    for row, (m, name) in enumerate([(model, "standard"), (robust_model, "robust")]):
        axes[row, col].imshow(saliency(m, x, t), cmap="Reds"); axes[row, col].set_xticks([]); axes[row, col].set_yticks([]); axes[row, col].grid(False)
        axes[row, col].set_title(f"{name} model, digit {t}", fontsize=8)
plt.suptitle("Input gradients of the standard and the PGD-trained model"); plt.tight_layout(); plt.show()

Two things are visible. The robust model's accuracy degrades far more slowly with $\epsilon$: in the full run the curve is nearly flat up to the training radius, and even after the single `FAST_RUN` epoch the robust model keeps a large fraction of its accuracy at $\epsilon = 0.15$ and $0.2$ where the standard model has collapsed. At the same time its clean accuracy ($\epsilon = 0$) is lower, by a modest amount in the full run and by more in the short `FAST_RUN` training, which is the trade-off in its purest form. The saliency maps of the robust model are also strikingly cleaner: they follow the strokes of the digit instead of speckling the background, because a model that cannot be moved by small perturbations of irrelevant pixels has, by construction, small gradients there. This is a known and somewhat surprising side effect of adversarial training: robustness and interpretability tend to arrive together.

> **Common pitfall.** Defences that merely *hide* the gradient (adding noise at test time, quantizing inputs, non-differentiable preprocessing) can look robust against gradient attacks and fall to adaptive or transfer attacks. Athalye et al. (2018) found that seven of the nine defences accepted at one conference (ICLR 2018) relied on such obfuscated gradients, and circumvented six of them completely and the seventh partially. Adversarial training is the exception because it changes the function itself, not the attacker's access to it.

### 4.7 Certified robustness

Empirical robustness against PGD is not a guarantee: a stronger attacker might exist. **Certified** defences prove that no perturbation within the budget changes the prediction. Two families are practical. *Interval bound propagation* and its relatives (Gowal et al., 2018; Zhang et al., 2018) propagate an interval or a linear bound for every activation through the network, given an interval around the input, and check that the lower bound of the true class's logit exceeds the upper bound of every other; training with these bounds in the loss yields networks that can be certified, at a substantial accuracy cost. *Randomized smoothing* (Cohen, Rosenfeld and Kolter, 2019) turns any classifier $f$ into a smoothed classifier $g(x) = \arg\max_c \Pr_{\delta \sim \mathcal{N}(0, \sigma^2 I)}[f(x + \delta) = c]$ and proves that if the top class has probability at least $p_A$ under the noise, then $g$ is constant within an $L_2$ ball of radius $\sigma\, \Phi^{-1}(p_A)$ around $x$. The probability is estimated by Monte Carlo with a confidence bound, which makes the certificate probabilistic but scalable to ImageNet-sized models; Exercise 5 implements the estimate.

## 5. Distribution shift and out-of-distribution detection

### 5.1 Accuracy and confidence under shift

Adversarial perturbations are the worst case; the *common* case is **distribution shift**, where the test inputs differ from the training distribution for benign reasons: a new camera, a different hospital, a rotated scanner, a noisier sensor. A model's accuracy under shift is easy to measure when labels are available, and the interesting quantity to measure alongside it is its **confidence**, the maximum softmax probability. A well-behaved model should become less confident as its inputs drift away from what it knows; a typical deep network does not.

We build three families of shifted test sets from the MNIST test tensor with pure tensor operations: rotations (bilinear, through torchvision's `transforms.v2.functional.rotate`), additive Gaussian noise, and translations by padding and cropping. For each severity level we record accuracy and mean confidence.

In [ ]:
def rotate_images(X, angle):
    return TF.rotate(X, float(angle), interpolation=InterpolationMode.BILINEAR)

def add_noise(X, sigma, seed=SEED):
    g = torch.Generator().manual_seed(seed)
    return (X + sigma * torch.randn(X.shape, generator=g)).clamp(0.0, 1.0)

def translate_images(X, dx, dy):
    """Shift by (dx, dy) pixels with zero fill (pad then crop; no wrap-around)."""
    H, W = X.shape[-2:]
    padded = F.pad(X, (abs(dx), abs(dx), abs(dy), abs(dy)))
    return padded[:, :, abs(dy) - dy: abs(dy) - dy + H, abs(dx) - dx: abs(dx) - dx + W]

def accuracy_and_confidence(model, X, y):
    probs = F.softmax(predict_logits(model, X), 1)
    return (probs.argmax(1) == y).float().mean().item(), probs.max(1).values.mean().item()

shift_families = {"rotation (degrees)": ([0, 15, 30, 45, 60, 90], lambda X, a: rotate_images(X, a)),
                  "Gaussian noise (std)": ([0.0, 0.1, 0.2, 0.3, 0.5], lambda X, s: add_noise(X, s)),
                  "translation (pixels)": ([0, 2, 4, 6, 8], lambda X, s: translate_images(X, s, s))}
shift_results = {}
fig, axes = plt.subplots(1, 3, figsize=(12, 3.6))
for ax, (name, (levels, fn)) in zip(axes, shift_families.items()):
    accs, confs = [], []
    for lvl in levels:
        acc, conf = accuracy_and_confidence(model, fn(X_test, lvl), y_test)
        accs.append(acc); confs.append(conf)
    shift_results[name] = (levels, accs, confs)
    ax.plot(levels, accs, "o-", label="accuracy"); ax.plot(levels, confs, "s--", label="mean confidence")
    ax.set_xlabel(name); ax.set_ylim(0, 1.02); ax.set_title(name.split(" (")[0]); ax.legend(fontsize=8)
plt.suptitle("Accuracy and confidence of the standard CNN under distribution shift"); plt.tight_layout(); plt.show()

# Show a few shifted examples.
fig, axes = plt.subplots(1, 5, figsize=(8, 2))
examples = [("clean", X_test[:1]), ("rot 45", rotate_images(X_test[:1], 45)), ("noise 0.3", add_noise(X_test[:1], 0.3)),
            ("shift 6", translate_images(X_test[:1], 6, 6)), ("rot 90", rotate_images(X_test[:1], 90))]
for ax, (name, img) in zip(axes, examples):
    ax.imshow(img.squeeze(), cmap="gray", vmin=0, vmax=1); ax.set_title(name, fontsize=9); ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
plt.tight_layout(); plt.show()

The two curves in each panel separate as the shift grows: accuracy falls, confidence falls much more slowly. Under a $45^\circ$ rotation the model may be wrong on a large fraction of digits while still assigning an average maximum probability well above what its accuracy would justify, and under strong noise the same pattern holds. This gap between confidence and accuracy is **miscalibration under shift**, and it is what makes silent failures possible: a monitoring system that only looks at the model's confidence would not notice that the input distribution had changed. Section 6 measures the gap precisely and Section 6.3 shows that the standard fix (temperature scaling) helps in-distribution but not under shift.

### 5.2 Out-of-distribution detection

Shifted digits are still digits. A stronger case is an input that has no correct label at all: a photograph of a shoe presented to a digit classifier. **Out-of-distribution (OOD) detection** asks the model to flag such inputs rather than classify them. With only a classifier and no OOD training data, two baseline scores are standard.

The **maximum softmax probability** (MSP, Hendrycks and Gimpel, 2017) uses the model's own confidence: in-distribution inputs tend to receive a higher maximum probability than OOD inputs. It works surprisingly often, but softmax saturates: a logit vector that is merely *large* produces a probability near one regardless of whether the input is familiar.

The **energy score** (Liu et al., 2020) avoids the saturation. For logits $f(x) \in \mathbb{R}^K$ and temperature $T$, define

$$
E(x) = -T \log \sum_{k=1}^{K} \exp\big( f_k(x) / T \big) .
$$

The motivation is that a discriminative classifier implicitly defines a joint density $p(x, k) \propto \exp(f_k(x)/T)$, so that $p(x) \propto \sum_k \exp(f_k(x)/T) = \exp(-E(x)/T)$: the negative energy is an unnormalized log-density of the input under the model's implicit generative view. Inputs with low energy (large logits overall) are "familiar", inputs with high energy are not. Unlike MSP, energy depends on the *magnitude* of the logits and not only on their ratios, and it is computed by a single `logsumexp`. We use $-E(x)$ as the score so that, like MSP, higher means more in-distribution.

The detection quality is measured by the **AUROC** of the score for separating in-distribution from OOD inputs: the probability that a random in-distribution input receives a higher score than a random OOD input, with 0.5 meaning no separation and 1.0 perfect separation. It is threshold-free, which is what we want when comparing scores. We use the FashionMNIST test set (clothing images with the same format as MNIST) as the OOD set and the $60^\circ$-rotated MNIST digits as a harder, "near-OOD" set.

In [ ]:
X_fashion, _ = load_split(datasets.FashionMNIST, train=False)
X_fashion = X_fashion[:N_TEST]
X_rot60 = rotate_images(X_test, 60)

def msp_score(logits):
    return F.softmax(logits, 1).max(1).values

def energy_score(logits, T=1.0):
    """Negative energy, T * logsumexp(f / T): higher = more in-distribution."""
    return T * torch.logsumexp(logits / T, 1)

def auroc_id_vs_ood(score_id, score_ood):
    labels = np.concatenate([np.ones(len(score_id)), np.zeros(len(score_ood))])
    scores = np.concatenate([score_id.detach().cpu().numpy(), score_ood.detach().cpu().numpy()])
    return roc_auc_score(labels, scores)

logits_id = predict_logits(model, X_test)
ood_sets = {"FashionMNIST": predict_logits(model, X_fashion), "MNIST rotated 60 deg": predict_logits(model, X_rot60)}

print(f"fraction of FashionMNIST images classified with confidence > 0.9: {(msp_score(ood_sets['FashionMNIST']) > 0.9).float().mean():.3f}")
print(f"\n{'OOD set':22s} {'AUROC (MSP)':>12s} {'AUROC (energy)':>15s}")
for name, logits_ood in ood_sets.items():
    print(f"{name:22s} {auroc_id_vs_ood(msp_score(logits_id), msp_score(logits_ood)):12.3f} "
          f"{auroc_id_vs_ood(energy_score(logits_id), energy_score(logits_ood)):15.3f}")

fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
axes[0].hist(msp_score(logits_id).cpu().numpy(), bins=30, alpha=0.6, label="MNIST test (ID)")
axes[0].hist(msp_score(ood_sets["FashionMNIST"]).cpu().numpy(), bins=30, alpha=0.6, label="FashionMNIST (OOD)")
axes[0].set_xlabel("max softmax probability"); axes[0].set_ylabel("count"); axes[0].set_title("MSP distributions"); axes[0].legend(fontsize=8)
axes[1].hist(energy_score(logits_id).cpu().numpy(), bins=30, alpha=0.6, label="MNIST test (ID)")
axes[1].hist(energy_score(ood_sets["FashionMNIST"]).cpu().numpy(), bins=30, alpha=0.6, label="FashionMNIST (OOD)")
axes[1].set_xlabel("negative energy"); axes[1].set_title("energy distributions"); axes[1].legend(fontsize=8)
for score_fn, name in [(msp_score, "MSP"), (energy_score, "energy")]:
    labels = np.concatenate([np.ones(len(logits_id)), np.zeros(len(X_fashion))])
    scores = np.concatenate([score_fn(logits_id).cpu().numpy(), score_fn(ood_sets["FashionMNIST"]).cpu().numpy()])
    fpr, tpr, _ = roc_curve(labels, scores)
    axes[2].plot(fpr, tpr, label=f"{name} (AUROC {roc_auc_score(labels, scores):.3f})")
axes[2].plot([0, 1], [0, 1], "k:"); axes[2].set_xlabel("false positive rate"); axes[2].set_ylabel("true positive rate")
axes[2].set_title("ROC: MNIST vs FashionMNIST"); axes[2].legend(fontsize=8)
plt.tight_layout(); plt.show()

The MSP histogram of the in-distribution set piles up near 1.0; the OOD histogram is broad but a sizeable fraction of clothing images still receive a confidence above 0.9, which is the saturation problem in action. Energy separates the two sets more cleanly, and its AUROC is typically a few points higher, because the logits of clothing images are smaller in magnitude even when their ratios happen to favour one class. Neither score is anywhere near perfect: an AUROC of 0.8 means that one in five random (digit, shoe) pairs is ranked the wrong way round. The rotated digits are a different kind of test: they share the low-level statistics of the training data, so which of the two OOD sets is harder depends on the model (with the fully trained model, rotated digits usually produce larger logits than clothing and are the harder set). This "near-OOD" regime, inputs that are almost but not quite in-distribution, remains an open problem. Two further families of detectors are worth knowing: feature-space methods, which model the training features with class-conditional Gaussians and score by Mahalanobis distance (Exercise 4), and methods that use *uncertainty* rather than confidence, such as the disagreement of an ensemble, which Section 6 provides.

> **Try it.** Change the temperature in `energy_score` to `T=10` and to `T=0.1`. Large $T$ makes the negative energy approach the mean logit plus the constant $T \log K$ (which does not affect the AUROC), small $T$ makes it approach the maximum logit; the original paper found $T = 1$ to be a robust choice.

Notebook 16 · Interpretability, Robustness and Uncertainty — part 3 (uncertainty and calibration, fairness/privacy/ethics, exercises, summary)

## 6. Uncertainty and calibration

### 6.1 Two kinds of not knowing

A prediction can be uncertain for two different reasons. **Aleatoric** uncertainty is inherent to the data: a badly written digit that could be a 4 or a 9 is ambiguous no matter how much data we collect, and the right answer is a probability split between the two classes. **Epistemic** uncertainty is the model's own ignorance: an input from a region where there was no training data, where the model's parameters are not pinned down and different plausible models would disagree. More data reduces epistemic uncertainty but not aleatoric uncertainty. The distinction matters because the tools differ: a softmax output can in principle express aleatoric uncertainty (if it is calibrated), but a single network has no way to express epistemic uncertainty, since it reports one answer with no notion of how different an equally good network's answer might be. Sections 6.4 and 6.5 obtain epistemic uncertainty from the disagreement between several predictors.

### 6.2 Calibration, reliability diagrams and expected calibration error

A probabilistic classifier is **calibrated** if its confidence means what it says: among all predictions made with confidence 0.8, about 80% should be correct. Formally, with $\hat p$ the predicted probability of the predicted class $\hat y$,

$$
\Pr\big( \hat y = y \mid \hat p = p \big) = p \qquad \text{for all } p \in [0, 1].
$$

Guo et al. (2017) showed that modern deep networks, unlike the smaller networks of the 1990s, are typically *overconfident*: their confidence exceeds their accuracy, a consequence of training to zero loss with high capacity. Calibration is assessed with a **reliability diagram**: bin the predictions by confidence, and plot the accuracy of each bin against its mean confidence; a calibrated model lies on the diagonal. The **expected calibration error** summarizes the diagram as the weighted average gap,

$$
\mathrm{ECE} = \sum_{b=1}^{B} \frac{|S_b|}{N}\, \big| \mathrm{acc}(S_b) - \mathrm{conf}(S_b) \big| ,
$$

where $S_b$ is the set of predictions in bin $b$. ECE is intuitive but depends on the binning and ignores the non-predicted classes; the **negative log-likelihood** (the test cross-entropy) is the complementary proper scoring rule that penalizes both miscalibration and inaccuracy, and we report both. We implement the statistics on the clean test set and on the $30^\circ$-rotated one, where Section 5 suggested the confidence gap is large.

In [ ]:
def calibration_stats(probs, labels, n_bins=15):
    """ECE and per-bin accuracy/confidence/fraction for predicted-class probabilities (N, K) and labels (N,)."""
    conf, pred = probs.max(1)
    correct = (pred == labels).float()
    edges = torch.linspace(0, 1, n_bins + 1)
    ece, bin_acc, bin_conf, bin_frac = 0.0, [], [], []
    for lo, hi in zip(edges[:-1], edges[1:]):
        mask = (conf > lo) & (conf <= hi)
        frac = mask.float().mean().item()
        if frac > 0:
            a, c = correct[mask].mean().item(), conf[mask].mean().item()
            ece += frac * abs(a - c)
        else:
            a = c = float("nan")
        bin_acc.append(a); bin_conf.append(c); bin_frac.append(frac)
    return ece, np.array(bin_acc), np.array(bin_conf), np.array(bin_frac), edges.cpu().numpy()

def nll(probs, labels):
    return F.nll_loss(torch.log(probs.clamp_min(1e-12)), labels).item()

def reliability_diagram(probs, labels, ax, title, n_bins=15):
    ece, acc, conf, frac, edges = calibration_stats(probs, labels, n_bins)
    centers, width = (edges[:-1] + edges[1:]) / 2, 1.0 / n_bins
    ok = frac > 0
    ax.bar(centers[ok], acc[ok], width=width, edgecolor="k", color="steelblue", alpha=0.8, label="accuracy in bin")
    ax.bar(centers[ok], (conf - acc)[ok], bottom=acc[ok], width=width, color="crimson", alpha=0.35, label="gap to confidence")
    ax.plot([0, 1], [0, 1], "k--", label="perfect calibration")
    ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.set_xlabel("confidence"); ax.set_ylabel("accuracy")
    ax.set_title(f"{title}\nECE = {ece:.3f}, NLL = {nll(probs, labels):.3f}", fontsize=9); ax.legend(fontsize=7, loc="upper left")
    return ece

X_rot30 = rotate_images(X_test, 30)
probs_test = F.softmax(logits_id, 1)
probs_rot30 = F.softmax(predict_logits(model, X_rot30), 1)
fig, axes = plt.subplots(1, 2, figsize=(9, 3.8))
reliability_diagram(probs_test, y_test, axes[0], "standard CNN, clean test set")
reliability_diagram(probs_rot30, y_test, axes[1], "standard CNN, rotated 30 degrees")
plt.tight_layout(); plt.show()

On the clean test set the model is close to the diagonal: MNIST is easy enough that most predictions sit in the top bin with a confidence near one and are indeed nearly always right, so the ECE is small. Under a $30^\circ$ rotation the bars fall below the diagonal across the whole range, the red gaps grow, and the ECE increases several-fold: the model keeps announcing confidence it no longer earns. A user of this model, say a system that only escalates predictions with confidence below 0.9 to a human, would escalate far too little on rotated inputs.

### 6.3 Temperature scaling

The simplest calibration fix (Guo et al., 2017) is to rescale the logits by a single learned scalar, the **temperature** $T > 0$:

$$
\hat p_k = \mathrm{softmax}\big( f(x) / T \big)_k = \frac{\exp(f_k(x)/T)}{\sum_j \exp(f_j(x)/T)} .
$$

Dividing by $T > 1$ flattens the distribution and reduces confidence; $T < 1$ sharpens it. Because all logits are divided by the same positive number, the arg-max, and therefore the accuracy, is unchanged: temperature scaling adjusts *only* calibration. The temperature is chosen to minimize the negative log-likelihood on a **held-out validation set** (never the test set, and never the training set, on which the model is overconfident by construction). The objective is a smooth one-dimensional problem, so we solve it with `torch.optim.LBFGS`, parametrizing $T = e^{\tau}$ to keep it positive.

In [ ]:
def fit_temperature(logits, labels, max_iter=200):
    """Fit T > 0 minimizing the cross-entropy of softmax(logits / T) on validation data with L-BFGS."""
    log_T = torch.zeros(1, requires_grad=True)
    opt = torch.optim.LBFGS([log_T], lr=0.1, max_iter=max_iter, line_search_fn="strong_wolfe")
    def closure():
        opt.zero_grad()
        loss = F.cross_entropy(logits / log_T.exp(), labels)
        loss.backward()
        return loss
    opt.step(closure)
    return log_T.exp().item()

logits_val = predict_logits(model, X_val)
T_opt = fit_temperature(logits_val, y_val)
print(f"fitted temperature T = {T_opt:.3f}  (validation NLL {F.cross_entropy(logits_val, y_val).item():.4f} -> "
      f"{F.cross_entropy(logits_val / T_opt, y_val).item():.4f})")

eval_sets = {"clean test": (logits_id, y_test), "rotated 30": (predict_logits(model, X_rot30), y_test),
             "rotated 45": (predict_logits(model, rotate_images(X_test, 45)), y_test)}
print(f"\n{'dataset':12s} {'accuracy':>9s} {'ECE before':>11s} {'ECE after':>10s} {'NLL before':>11s} {'NLL after':>10s}")
for name, (lg, yy) in eval_sets.items():
    p_before, p_after = F.softmax(lg, 1), F.softmax(lg / T_opt, 1)
    print(f"{name:12s} {(lg.argmax(1) == yy).float().mean().item():9.3f} {calibration_stats(p_before, yy)[0]:11.3f} "
          f"{calibration_stats(p_after, yy)[0]:10.3f} {nll(p_before, yy):11.3f} {nll(p_after, yy):10.3f}")

fig, axes = plt.subplots(1, 2, figsize=(9, 3.8))
reliability_diagram(F.softmax(logits_id / T_opt, 1), y_test, axes[0], f"temperature-scaled (T = {T_opt:.2f}), clean test")
reliability_diagram(F.softmax(eval_sets["rotated 30"][0] / T_opt, 1), y_test, axes[1], f"temperature-scaled (T = {T_opt:.2f}), rotated 30")
plt.tight_layout(); plt.show()

For a fully trained network the fitted temperature is typically a little above one (the model is overconfident on validation data, and dividing the logits softens it); for the briefly trained `FAST_RUN` model it can come out *below* one, because a network that has not yet fitted its training data is underconfident and needs sharpening. In both cases the in-distribution ECE and NLL improve, which is what the temperature was fitted for. The rotated sets tell the important half of the story: a single temperature fitted on clean validation data cannot know how the model's confidence behaves on shifted inputs, so the effect there is small and can even reverse, with the ECE on rotated digits getting *worse* (Ovadia et al., 2019, documented this systematically). Temperature scaling is a cheap, safe post-processing step that should be applied to almost every deployed classifier, and it is not a substitute for epistemic uncertainty.

### 6.4 Monte Carlo dropout

Dropout (Notebook 04 · Training Deep Networks: Optimization, Initialization, Normalization, Regularization) was introduced as a regularizer and is switched off at test time. Gal and Ghahramani (2016) showed that keeping it *on* at test time and averaging over several stochastic forward passes can be interpreted as approximate Bayesian inference: each dropout mask corresponds to a sample from an approximate posterior over networks, and the spread of their predictions is a measure of epistemic uncertainty. **MC dropout** therefore requires no retraining, only a model that contains dropout layers (ours has one before the output layer) and $S$ forward passes per input.

From the $S$ probability vectors $p^{(s)}$ we form the predictive distribution $\bar p = \frac{1}{S}\sum_s p^{(s)}$ and two uncertainty measures: the **predictive entropy** $\mathcal{H}[\bar p]$, which captures total uncertainty, and the **mutual information** $\mathcal{H}[\bar p] - \frac{1}{S}\sum_s \mathcal{H}[p^{(s)}]$, which captures only the *disagreement* between the samples, that is the epistemic part. A blurry digit gives high entropy but low mutual information (all samples agree it is ambiguous); a shoe gives high mutual information (samples disagree about which digit it is). We switch on only the dropout modules, leaving everything else in evaluation mode, which matters for models with batch normalization.

In [ ]:
def enable_dropout(model):
    """Evaluation mode, except that Dropout layers keep sampling masks."""
    model.eval()
    for m in model.modules():
        if isinstance(m, nn.Dropout):
            m.train()

@torch.no_grad()
def mc_dropout_probs(model, X, n_samples=20, batch_size=512, seed=SEED):
    """Stochastic forward passes with dropout active. Returns probabilities of shape (n_samples, N, K)."""
    enable_dropout(model)
    torch.manual_seed(seed)
    out = []
    for i in range(0, len(X), batch_size):
        xb = X[i:i + batch_size].to(device)
        out.append(torch.stack([F.softmax(model(xb), 1) for _ in range(n_samples)]).cpu())
    model.eval()
    return torch.cat(out, dim=1)

def entropy(p):
    return -(p * torch.log(p.clamp_min(1e-12))).sum(-1)

def mutual_information(samples):
    """samples: (S, N, K). MI = H[mean p] - mean_s H[p_s]."""
    return entropy(samples.mean(0)) - entropy(samples).mean(0)

MC_SAMPLES = 20
mc_test = mc_dropout_probs(model, X_test, MC_SAMPLES)
mc_fashion = mc_dropout_probs(model, X_fashion, MC_SAMPLES)
mc_rot30 = mc_dropout_probs(model, X_rot30, MC_SAMPLES)
p_mc_test = mc_test.mean(0)
print(f"MC dropout ({MC_SAMPLES} samples): test accuracy {(p_mc_test.argmax(1) == y_test).float().mean():.4f}, "
      f"ECE {calibration_stats(p_mc_test, y_test)[0]:.3f} (clean), {calibration_stats(mc_rot30.mean(0), y_test)[0]:.3f} (rotated 30)")
print(f"mean predictive entropy : MNIST {entropy(p_mc_test).mean():.3f} | FashionMNIST {entropy(mc_fashion.mean(0)).mean():.3f}")
print(f"mean mutual information : MNIST {mutual_information(mc_test).mean():.3f} | FashionMNIST {mutual_information(mc_fashion).mean():.3f}")
print(f"AUROC MNIST vs FashionMNIST with predictive entropy : {auroc_id_vs_ood(-entropy(p_mc_test), -entropy(mc_fashion.mean(0))):.3f}")
print(f"AUROC MNIST vs FashionMNIST with mutual information : {auroc_id_vs_ood(-mutual_information(mc_test), -mutual_information(mc_fashion)):.3f}")

Both uncertainty measures are markedly larger on FashionMNIST than on MNIST. The predictive entropy separates the two sets about as well as the confidence-based scores of Section 5, while the mutual information alone is weaker here: a single dropout layer before the output produces a rather limited family of "sampled networks", so the disagreement signal is small and noisy. Its quality would improve with dropout in more layers, at the cost of the model's accuracy. Note also the price: $S = 20$ forward passes per input, which is why MC dropout is often used with a small $S$ or replaced by the next method when latency matters.

### 6.5 Deep ensembles

The most reliable practical method for epistemic uncertainty is also the most obvious: train $M$ networks from different random initializations (and data orders) and average their predictions (Lakshminarayanan et al., 2017). Different initializations end in different minima that agree on the training distribution and disagree away from it, which is exactly the behaviour we want. The predictive distribution is again the mean of the members' probabilities and the mutual information measures their disagreement. Deep ensembles improve accuracy, calibration and OOD detection at the same time and were the strongest method in the large-scale comparison of Ovadia et al. (2019); their cost is $M$ times the training and inference compute, which is why $M$ is usually between 3 and 5. We train two more copies of our CNN (with different seeds and data orders) and combine them with the base model.

In [ ]:
ENSEMBLE_SIZE = 3
ensemble = [model]
for k in range(1, ENSEMBLE_SIZE):
    torch.manual_seed(SEED + 100 * k)
    loader_k = DataLoader(TensorDataset(X_train, y_train), batch_size=128, shuffle=True, num_workers=0,
                          pin_memory=(device.type == "cuda"), generator=torch.Generator().manual_seed(SEED + 100 * k))
    ensemble.append(fit(SmallCNN().to(device), loader_k, EPOCHS, name=f"ensemble member {k + 1}", seed=SEED + 100 * k))

def ensemble_probs(models, X):
    return torch.stack([F.softmax(predict_logits(m, X), 1) for m in models])     # (M, N, K)

ens_test, ens_fashion, ens_rot30 = (ensemble_probs(ensemble, X_) for X_ in (X_test, X_fashion, X_rot30))
print(f"\nindividual test accuracies: {[round((p.argmax(1) == y_test).float().mean().item(), 4) for p in ens_test]}")
print(f"ensemble test accuracy    : {(ens_test.mean(0).argmax(1) == y_test).float().mean():.4f}")
print(f"AUROC MNIST vs FashionMNIST with ensemble mutual information: "
      f"{auroc_id_vs_ood(-mutual_information(ens_test), -mutual_information(ens_fashion)):.3f}")

### 6.6 Putting the methods side by side

We now compare the four predictors on the same criteria: accuracy and NLL on the clean test set, calibration (ECE) on the clean and on the rotated test set, and OOD detection of FashionMNIST by two scores, the confidence of the predictive distribution and its entropy, plus the mutual information where it exists. The table is the kind of evidence one should produce before choosing an uncertainty method for a deployment.

In [ ]:
def summarize(name, p_test, p_rot, p_fashion, samples_test=None, samples_fashion=None):
    row = {"method": name,
           "acc": (p_test.argmax(1) == y_test).float().mean().item(),
           "NLL": nll(p_test, y_test),
           "ECE clean": calibration_stats(p_test, y_test)[0],
           "ECE rot30": calibration_stats(p_rot, y_test)[0],
           "AUROC conf": auroc_id_vs_ood(p_test.max(1).values, p_fashion.max(1).values),
           "AUROC entropy": auroc_id_vs_ood(-entropy(p_test), -entropy(p_fashion)),
           "AUROC MI": (auroc_id_vs_ood(-mutual_information(samples_test), -mutual_information(samples_fashion))
                        if samples_test is not None else float("nan"))}
    return row

rows = [summarize("single model", probs_test, probs_rot30, F.softmax(ood_sets["FashionMNIST"], 1)),
        summarize(f"temperature scaling (T={T_opt:.2f})", F.softmax(logits_id / T_opt, 1),
                  F.softmax(eval_sets["rotated 30"][0] / T_opt, 1), F.softmax(ood_sets["FashionMNIST"] / T_opt, 1)),
        summarize(f"MC dropout ({MC_SAMPLES} samples)", mc_test.mean(0), mc_rot30.mean(0), mc_fashion.mean(0), mc_test, mc_fashion),
        summarize(f"deep ensemble (M={ENSEMBLE_SIZE})", ens_test.mean(0), ens_rot30.mean(0), ens_fashion.mean(0), ens_test, ens_fashion)]

cols = ["acc", "NLL", "ECE clean", "ECE rot30", "AUROC conf", "AUROC entropy", "AUROC MI"]
print(f"{'method':32s}" + "".join(f"{c:>15s}" for c in cols))
for r in rows:
    print(f"{r['method']:32s}" + "".join(f"{r[c]:15.3f}" for c in cols))

The pattern that this table usually shows, and that holds far beyond MNIST, is the following. Temperature scaling fixes in-distribution calibration at zero cost but does little for the shifted set or for OOD detection: a single global rescaling of the logits changes how confident the model *sounds*, not what it knows, and a temperature fitted for calibration (close to one) barely changes the ranking of inputs by confidence. (A very large temperature can sharpen the MSP ranking somewhat, the observation behind ODIN, Liang et al., 2018, but that is a different use of $T$ from calibration.) MC dropout adds a genuine epistemic signal (the mutual information column) and improves calibration under shift somewhat. The deep ensemble is the best or near-best on every column at once: higher accuracy, lower NLL, better calibration on the rotated set, and the strongest OOD detection through disagreement. Its only drawback is cost, and a good deal of research (snapshot ensembles, SWAG, batch ensembles, Laplace approximations) aims at approximating it more cheaply. Whichever method is chosen, the evaluation must include a shifted and an OOD set, because in-distribution metrics do not predict the behaviour that matters in deployment.

> **Going deeper.** The mutual-information column is the operational meaning of "epistemic": it is zero when all members agree, however uncertain they are. For an ambiguous digit, entropy is high and mutual information is low (aleatoric); for a shoe, both are high. Systems that need to distinguish "the input is ambiguous" from "the input is foreign" should log both.

## 7. Fairness, privacy and ethics

### 7.1 Bias in data and fairness metrics

A model trained on historical decisions learns the regularities of those decisions, including the ones we would rather it did not. Consider an automated screening model for job applicants. The historical labels ("was hired") reflect past human judgement; the features may include proxies for group membership (a neighbourhood, a credential that one group had less access to); and simply removing the protected attribute from the inputs, so-called **fairness through unawareness**, does not help when other features carry the same information. Because such data cannot be shared, we generate a synthetic dataset in which the truth is known: a latent *qualification* that is independent of the group, a fair test score, a credential that is structurally harder for group 1 to obtain, a neighbourhood variable correlated with the group, and historical hiring labels that were biased against group 1.

The two most common group fairness criteria, for a binary decision $\hat Y$, a binary label $Y$ and a protected attribute $A$, are

- **Demographic parity** (statistical parity): the selection rate is the same for both groups, $\Pr(\hat Y = 1 \mid A = 0) = \Pr(\hat Y = 1 \mid A = 1)$. It does not look at the labels at all.
- **Equalized odds** (Hardt, Price and Srebro, 2016): the true positive rate and the false positive rate are the same for both groups, $\Pr(\hat Y = 1 \mid Y = y, A = 0) = \Pr(\hat Y = 1 \mid Y = y, A = 1)$ for $y \in \{0, 1\}$. Its relaxation **equal opportunity** requires this only for $y = 1$ (qualified candidates are selected at equal rates).

We report both as *gaps* between the groups, together with the accuracy per group. A third family, calibration within groups, was at the centre of the COMPAS debate: Chouldechova (2017) and Kleinberg, Mullainathan and Raghavan (2017) proved that when the base rates $\Pr(Y = 1 \mid A)$ differ between groups, no non-trivial classifier can satisfy calibration and equalized odds at the same time. Fairness is therefore not a property one can switch on; it is a choice among incompatible criteria that must be made explicitly for each application.

In [ ]:
def make_hiring_dataset(n, seed=SEED):
    """Synthetic applicant data with a known ground truth. Returns features, historical labels, group, true qualification."""
    rng = np.random.default_rng(seed)
    group = rng.integers(0, 2, n)                                            # protected attribute A
    skill = rng.normal(0.0, 1.0, n)                                          # latent qualification, independent of A
    test_score = skill + rng.normal(0.0, 0.6, n)                             # a fair measurement
    credential = skill - 0.8 * group + rng.normal(0.0, 0.6, n)               # structurally harder to obtain for group 1
    neighbourhood = group + rng.normal(0.0, 0.7, n)                          # correlated with A, unrelated to skill
    hired = (skill - 0.5 * group + rng.normal(0.0, 0.5, n) > 0.3).astype(int)   # historical decisions, biased against group 1
    qualified = (skill > 0.3).astype(int)                                    # what an unbiased decision would be based on
    X = np.stack([test_score, credential, neighbourhood], axis=1)
    return X, hired, group, qualified

def group_metrics(y_true, y_pred, group):
    """Per-group selection rate, TPR, FPR and accuracy, plus the demographic-parity and equalized-odds gaps."""
    rows = {}
    for g in (0, 1):
        m = group == g
        rows[g] = {"selection rate": y_pred[m].mean(), "TPR": y_pred[m & (y_true == 1)].mean(),
                   "FPR": y_pred[m & (y_true == 0)].mean(), "accuracy": (y_pred[m] == y_true[m]).mean()}
    dp_gap = abs(rows[0]["selection rate"] - rows[1]["selection rate"])
    eo_gap = max(abs(rows[0]["TPR"] - rows[1]["TPR"]), abs(rows[0]["FPR"] - rows[1]["FPR"]))
    return rows, dp_gap, eo_gap

def print_metrics(title, y_true, y_pred, group):
    rows, dp_gap, eo_gap = group_metrics(y_true, y_pred, group)
    print(f"{title}")
    print(f"   {'':10s}" + "".join(f"{k:>16s}" for k in rows[0]))
    for g in (0, 1):
        print(f"   group {g:<4d}" + "".join(f"{v:16.3f}" for v in rows[g].values()))
    print(f"   demographic parity gap = {dp_gap:.3f} | equalized odds gap = {eo_gap:.3f} | overall accuracy = {(y_pred == y_true).mean():.3f}")

X_fair, y_hist, group, qualified = make_hiring_dataset(20_000)
n_tr = 12_000
clf = LogisticRegression().fit(X_fair[:n_tr], y_hist[:n_tr])          # "unaware": the group is not an input
scores = clf.predict_proba(X_fair[n_tr:])[:, 1]
y_pred = (scores >= 0.5).astype(int)
g_te, y_hist_te, qual_te = group[n_tr:], y_hist[n_tr:], qualified[n_tr:]

print(f"base rates of the historical label: group 0 {y_hist[group == 0].mean():.3f}, group 1 {y_hist[group == 1].mean():.3f}; "
      f"of true qualification: group 0 {qualified[group == 0].mean():.3f}, group 1 {qualified[group == 1].mean():.3f}\n")
print_metrics("Logistic regression (threshold 0.5), evaluated against the HISTORICAL labels:", y_hist_te, y_pred, g_te)
print()
print_metrics("The same predictions, evaluated against TRUE qualification:", qual_te, y_pred, g_te)

Even though the group is not an input, the model selects group 1 at a much lower rate and, judged against true qualification, misses many more qualified candidates from group 1 (a lower TPR): it has reconstructed the group from the credential and neighbourhood proxies and reproduced the historical bias. Judged against the historical labels the model looks reasonably accurate for both groups, which is exactly how such a system passes a naive validation: the labels share the bias.

The simplest mitigation is **post-processing**: keep the model and choose a separate decision threshold per group so that a chosen criterion holds (Hardt et al., 2016). Below we pick the threshold for group 1 so that its true positive rate (against the historical labels, the only ones a practitioner would have) matches that of group 0, i.e. we enforce equal opportunity, and observe what happens to the other metrics.

In [ ]:
def threshold_for_tpr(scores, y, target_tpr):
    """Largest threshold whose TPR on (scores, y) is at least target_tpr."""
    pos_scores = np.sort(scores[y == 1])[::-1]
    k = int(np.ceil(target_tpr * len(pos_scores)))
    return pos_scores[min(max(k, 1), len(pos_scores)) - 1]

rows0, _, _ = group_metrics(y_hist_te, y_pred, g_te)
t1 = threshold_for_tpr(scores[g_te == 1], y_hist_te[g_te == 1], rows0[0]["TPR"])
y_pred_eo = np.where(g_te == 0, scores >= 0.5, scores >= t1).astype(int)
print(f"group-specific threshold for group 1: {t1:.3f} (group 0 keeps 0.5)\n")
print_metrics("Equal-opportunity post-processing, evaluated against the HISTORICAL labels:", y_hist_te, y_pred_eo, g_te)
print()
print_metrics("The same predictions, evaluated against TRUE qualification:", qual_te, y_pred_eo, g_te)

Lowering the threshold for group 1 equalizes the true positive rates and shrinks the demographic-parity gap, at the cost of a higher false positive rate for group 1 and a small loss of accuracy against the historical labels. Against the true qualification, which is what the organization presumably cares about, the adjusted decisions are in fact *more* accurate for group 1, because the historical labels were biased. This is the general shape of fairness interventions: they trade off criteria that cannot all hold, the "right" trade-off depends on which label one believes, and the decision must be made and documented by people, not left to the optimizer. Other intervention points are pre-processing (reweighting or repairing the training data) and in-processing (adding a fairness constraint or penalty to the loss); post-processing is the least invasive and works with any trained model.

### 7.2 Privacy: what a model remembers

Models memorize. A language model can emit training sequences verbatim (Carlini et al., 2021), and even a classifier leaks whether a specific record was in its training set through **membership inference**: the model's loss on training points is systematically lower than on unseen points, and an attacker who can query confidences can exploit that difference. For data about people, this is a privacy breach even when the model never outputs raw data.

**Differential privacy** (Dwork et al., 2006) is the formal answer. A randomized training algorithm $\mathcal{M}$ is $(\varepsilon, \delta)$-differentially private if for any two datasets $D, D'$ that differ in one record and any set of outputs $S$,

$$
\Pr[\mathcal{M}(D) \in S] \;\le\; e^{\varepsilon}\, \Pr[\mathcal{M}(D') \in S] + \delta .
$$

Nothing that can be learned from the trained model depends much on whether any single person's record was included. **DP-SGD** (Abadi et al., 2016) makes ordinary training satisfy this guarantee with two modifications of each step: the gradient of *each example* is clipped to a maximum norm $C$ (bounding any one record's influence), and Gaussian noise with standard deviation $\sigma C$ is added to the summed gradient before the parameter update. A *privacy accountant* tracks how the $(\varepsilon, \delta)$ budget accumulates over the steps. The price is accuracy (the noise is a strong regularizer) and compute (per-example gradients are expensive, though `torch.func` and the Opacus library make them practical). Values of $\varepsilon$ in the single digits are considered meaningful protection; the reported $\varepsilon$ should be read together with $\delta$ (usually $1/N$) and the number of epochs. Differential privacy protects the *training data*; it does not address fairness, and the noise can in fact hurt minority groups disproportionately, which is one more reason to measure per-group metrics after any such intervention.

### 7.3 Documentation and responsible deployment

Most of the failures of Section 1 were failures of *communication* as much as of modelling: the systems were used on populations, inputs or decisions for which nobody had checked them. Two documentation practices have become standard.

**Datasheets for datasets** (Gebru et al., 2021) accompany a dataset with structured answers about its motivation, composition (who is in it, who is missing, how labels were produced), collection process, preprocessing, recommended uses and, importantly, uses it should not be put to. **Model cards** (Mitchell et al., 2019) do the same for a trained model: intended use and out-of-scope uses, the evaluation data, metrics disaggregated by relevant groups and conditions (the table of Section 6.6 and the per-group table of Section 7.1 are exactly the kind of content they call for), known limitations, and ethical considerations. Notebook 15 · Deployment and Model Optimization contains a model card template.

A deployment checklist that summarizes this notebook:

1. Report accuracy *and* calibration (ECE, NLL) on the test set, and on at least one shifted test set that resembles the deployment conditions.
2. Measure OOD detection (AUROC against a plausible foreign distribution) and decide what the system does when the score says "foreign": abstain, escalate, or flag.
3. Run attribution sanity checks and inspect explanations for a sample of predictions, looking specifically for shortcuts.
4. State the threat model: is adversarial robustness required, at what budget, and was it evaluated against PGD or a stronger adaptive attack?
5. Disaggregate every metric by the groups the system affects; choose and document the fairness criterion; revisit it with domain experts.
6. Decide whether the training data is sensitive, and if so whether differential privacy or access controls are needed; test for memorization.
7. Write the datasheet and the model card, including out-of-scope uses.
8. Monitor inputs and outputs after deployment for drift (Notebook 15), and define the rollback procedure.
9. Keep a human in the loop for high-stakes decisions and make the uncertainty visible to them.
10. Record which of the above were skipped and why; a checklist is only useful when its gaps are visible.

## Exercises

**Exercise 1 (warm-up).** Guided backpropagation (Springenberg et al., 2015) modifies the backward pass of every ReLU so that only *positive* gradients are propagated: $\partial L / \partial x = \mathbf{1}[x > 0]\, \mathbf{1}[\partial L / \partial y > 0]\, \partial L / \partial y$. Implement it with `register_full_backward_hook` on every `nn.ReLU` module of the CNN, compare the resulting maps with plain saliency on the three example digits, and run the model randomization test of Section 2.3 on it. What does the test say?

**Solution to Exercise 1**

In [ ]:
def guided_backprop(model, x, target):
    """Input gradient with ReLU backward passes that clamp negative gradients to zero (signed map)."""
    hooks = [m.register_full_backward_hook(lambda mod, grad_in, grad_out: (torch.clamp(grad_in[0], min=0.0),))
             for m in model.modules() if isinstance(m, nn.ReLU)]
    try:
        model.eval()
        x = x.clone().requires_grad_(True)
        score = model(x)[0, target]
        grad, = torch.autograd.grad(score, x)
    finally:
        for h in hooks:
            h.remove()
    return grad.squeeze().cpu().numpy()

fig, axes = plt.subplots(3, 3, figsize=(6, 6))
gbp_corr, sal_corr = [], []
for col, idx in enumerate(example_indices):
    x = X_test[idx:idx + 1].to(device); t = int(y_test[idx])
    sal, gbp, gbp_rand = saliency(model, x, t), guided_backprop(model, x, t), guided_backprop(random_model, x, t)
    gbp_corr.append(rank_correlation(gbp, gbp_rand))
    sal_corr.append(rank_correlation(sal, saliency(random_model, x, t)))
    for row, (m, name) in enumerate([(sal, "saliency"), (np.abs(gbp), "guided backprop"), (np.abs(gbp_rand), "guided backprop, random model")]):
        axes[row, col].imshow(m, cmap="Reds"); axes[row, col].set_xticks([]); axes[row, col].set_yticks([]); axes[row, col].grid(False)
        axes[row, col].set_title(f"{name} ({t})", fontsize=8)
plt.tight_layout(); plt.show()
print(f"rank correlation trained vs random model:  saliency {np.mean(sal_corr):.3f}   guided backprop {np.mean(gbp_corr):.3f}")

Guided backpropagation produces the cleanest-looking maps of all, sharp strokes with no background speckle, and that is precisely the problem: the map of the *randomly initialized* network looks much the same, and its rank correlation with the trained model's map is noticeably higher than for plain saliency (the gap widens with a fully trained model, whose plain gradients are more specific). By discarding negative gradients at every layer, guided backprop essentially reconstructs the input's edges and ignores what the model computes; it fails the randomization test, as Adebayo et al. (2018) reported. Visual appeal is not evidence of faithfulness.

**Exercise 2.** The Carlini–Wagner attack (2017) minimizes the size of the perturbation subject to misclassification, instead of maximizing the loss within a fixed budget. Implement a simplified untargeted $L_2$ version: parametrize $x^{\text{adv}} = \tfrac{1}{2}(\tanh w + 1)$ so that it always lies in $[0, 1]$, and minimize $\|x^{\text{adv}} - x\|_2^2 + c \cdot \max\big( Z_y - \max_{j \ne y} Z_j,\, -\kappa \big)$ with Adam, where $Z$ are the logits. Run it on 64 test images, report the success rate and the mean $L_2$ norm of the successful perturbations, and compare the norms with those of PGD at $\epsilon = 0.15$.

**Solution to Exercise 2**

In [ ]:
def cw_l2_attack(model, x, y, c=2.0, kappa=0.0, steps=150, lr=0.05):
    """Simplified Carlini-Wagner L2 attack (untargeted). Returns the smallest successful adversarial examples found
    (the clean image where the attack failed) and their L2 distances (inf where it failed)."""
    model.eval()
    w = torch.atanh((2 * x - 1).clamp(-1 + 1e-6, 1 - 1e-6)).detach().requires_grad_(True)
    opt = torch.optim.Adam([w], lr=lr)
    best_adv, best_l2 = x.clone(), torch.full((len(x),), float("inf"), device=x.device)
    onehot = F.one_hot(y, 10).bool()
    for _ in range(steps):
        x_adv = (torch.tanh(w) + 1) / 2
        logits = model(x_adv)
        z_y = logits[onehot]
        z_other = logits.masked_fill(onehot, float("-inf")).max(1).values
        margin = torch.clamp(z_y - z_other, min=-kappa)
        l2 = (x_adv - x).flatten(1).pow(2).sum(1)
        loss = (l2 + c * margin).sum()
        opt.zero_grad(); loss.backward(); opt.step()
        with torch.no_grad():
            improved = (logits.argmax(1) != y) & (l2 < best_l2)
            best_l2 = torch.where(improved, l2, best_l2)
            best_adv[improved] = x_adv[improved].detach()
    return best_adv, best_l2.sqrt()

xb, yb = X_test[:64].to(device), y_test[:64].to(device)
t0 = time.time()
cw_adv, cw_dist = cw_l2_attack(model, xb, yb)
success = torch.isfinite(cw_dist)
pgd_adv = pgd(model, xb, yb, eps=0.15, steps=10)
with torch.no_grad():
    pgd_success = model(pgd_adv).argmax(1) != yb
pgd_dist = (pgd_adv - xb).flatten(1).norm(dim=1)
mean_cw = cw_dist[success].mean().item() if success.any() else float("nan")
print(f"C&W-L2 : success rate {success.float().mean():.3f}, mean L2 of successful perturbations {mean_cw:.3f} ({time.time() - t0:.1f}s)")
print(f"PGD-Linf (eps 0.15): success rate {pgd_success.float().mean():.3f}, mean L2 norm of perturbations {pgd_dist.mean():.3f}")

if success.any():
    k = int(success.nonzero()[0])
    fig, axes = plt.subplots(1, 3, figsize=(7.5, 2.8))
    for ax, img, name in zip(axes, (xb[k], cw_adv[k], pgd_adv[k]), ("clean", f"C&W (L2 = {cw_dist[k]:.2f})", f"PGD (L2 = {pgd_dist[k]:.2f})")):
        ax.imshow(img.squeeze().cpu(), cmap="gray", vmin=0, vmax=1); ax.set_title(name, fontsize=9); ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
    plt.tight_layout(); plt.show()
else:
    print("no C&W example succeeded within this budget; increase c or the number of steps")

The C&W attack finds perturbations whose $L_2$ norm is a fraction of the PGD perturbation's, because it optimizes the norm directly instead of spending the whole $\epsilon$ budget on every pixel; the price is many more optimization steps and a constant $c$ that in the full attack is tuned by binary search per image. Minimum-norm attacks are the right tool for asking "how close is the nearest decision boundary?", while fixed-budget attacks like PGD answer "is the model robust at radius $\epsilon$?". Both should be in an evaluation, and $L_2$ and $L_\infty$ threat models are genuinely different: a model robust under one is not automatically robust under the other.

**Exercise 3.** Compare the calibration of the deep ensemble with that of a single model under increasing shift: draw reliability diagrams for both on the clean test set and on the $30^\circ$ and $60^\circ$ rotated sets, and tabulate ECE and accuracy. Does the ensemble stay calibrated when it becomes inaccurate?

**Solution to Exercise 3**

In [ ]:
sets = {"clean": X_test, "rotated 30": X_rot30, "rotated 60": X_rot60}     # X_rot60 was built in Section 5.2
fig, axes = plt.subplots(2, 3, figsize=(12, 7))
print(f"{'set':12s} {'single acc':>11s} {'single ECE':>11s} {'ensemble acc':>13s} {'ensemble ECE':>13s}")
for col, (name, X_) in enumerate(sets.items()):
    p_single = F.softmax(predict_logits(model, X_), 1)
    p_ens = ensemble_probs(ensemble, X_).mean(0)
    e_s = reliability_diagram(p_single, y_test, axes[0, col], f"single model, {name}")
    e_e = reliability_diagram(p_ens, y_test, axes[1, col], f"ensemble, {name}")
    print(f"{name:12s} {(p_single.argmax(1) == y_test).float().mean():11.3f} {e_s:11.3f} "
          f"{(p_ens.argmax(1) == y_test).float().mean():13.3f} {e_e:13.3f}")
plt.tight_layout(); plt.show()

Both predictors lose accuracy as the rotation grows, but the ensemble loses *confidence* along with it, so its reliability bars stay closer to the diagonal and its ECE grows more slowly; the single model's confidence barely moves while its accuracy falls, which is the overconfidence-under-shift failure of Section 5. The ensemble is not perfectly calibrated at $60^\circ$ either (three members trained on the same upright digits still share many blind spots), but it fails more gracefully, and gracefulness is what a downstream decision system needs.

**Exercise 4.** Implement the Mahalanobis OOD score of Lee et al. (2018): fit one Gaussian per class to the penultimate features (the 128-dimensional `fc1` activations) of the training set with a shared covariance matrix, and score an input by its negative squared Mahalanobis distance to the closest class mean. Compare its AUROC with MSP and energy on FashionMNIST and on the $60^\circ$-rotated digits.

**Solution to Exercise 4**

In [ ]:
feat_train = extract_features(model, X_train[:N_PROBE], "fc1").double()
y_feat = y_train[:N_PROBE]
class_means = torch.stack([feat_train[y_feat == c].mean(0) for c in range(10)])           # (10, 128)
centered = feat_train - class_means[y_feat]
shared_cov = centered.T @ centered / len(feat_train)
precision = torch.linalg.inv(shared_cov + 1e-4 * torch.eye(shared_cov.shape[0], dtype=shared_cov.dtype))

def mahalanobis_score(X_):
    """Negative squared Mahalanobis distance to the nearest class mean (higher = more in-distribution)."""
    f = extract_features(model, X_, "fc1").double()
    diff = f[:, None, :] - class_means[None, :, :]                                        # (N, 10, 128)
    d2 = torch.einsum("nci,ij,ncj->nc", diff, precision, diff)
    return -d2.min(1).values.float()

print(f"{'OOD set':22s} {'MSP':>8s} {'energy':>8s} {'Mahalanobis':>12s}")
for name, X_ood, lg_ood in [("FashionMNIST", X_fashion, ood_sets["FashionMNIST"]), ("MNIST rotated 60", X_rot60, ood_sets["MNIST rotated 60 deg"])]:
    print(f"{name:22s} {auroc_id_vs_ood(msp_score(logits_id), msp_score(lg_ood)):8.3f} "
          f"{auroc_id_vs_ood(energy_score(logits_id), energy_score(lg_ood)):8.3f} "
          f"{auroc_id_vs_ood(mahalanobis_score(X_test), mahalanobis_score(X_ood)):12.3f}")

The Mahalanobis score looks at *where* an input lands in feature space rather than at how the output layer reacts, and features of foreign inputs tend to fall far from every class cluster even when the logits happen to be large. On FashionMNIST it is usually the strongest of the three scores; on rotated digits, whose features drift but remain closer to the clusters, the advantage is smaller. The original paper adds input preprocessing and combines several layers with a small logistic-regression weighting; the single-layer version here already shows the main idea, that the training features are a density model of the familiar, and that a classifier's confidence is not.

**Exercise 5 (challenge).** Implement the certification estimate of randomized smoothing (Cohen et al., 2019). Fine-tune (or train) a copy of the CNN with Gaussian noise augmentation of standard deviation $\sigma = 0.25$ so that the base classifier is accurate on noisy inputs, then for a few test images: (a) sample $n_0$ noisy copies to guess the top class $c_A$; (b) sample $n$ more copies and count how often $c_A$ is predicted; (c) compute a one-sided lower confidence bound $\underline{p_A}$ on that frequency; (d) if $\underline{p_A} > \tfrac{1}{2}$, certify the $L_2$ radius $R = \sigma\, \Phi^{-1}(\underline{p_A})$, otherwise abstain. Compare the certified radii obtained with the noise-trained model and with the standard model.

**Solution to Exercise 5**

In [ ]:
SIGMA = 0.25
N_CERT = 200 if FAST_RUN else 1_000
torch.manual_seed(SEED + 7)
smooth_model = fit(SmallCNN().to(device), train_loader, EPOCHS, noise_std=SIGMA, name="noise-augmented CNN", seed=SEED + 7)
print(f"accuracy on noisy test images (sigma = {SIGMA}): standard {evaluate(model, add_noise(X_test, SIGMA), y_test)[1]:.3f}, "
      f"noise-trained {evaluate(smooth_model, add_noise(X_test, SIGMA), y_test)[1]:.3f}")

def lower_confidence_bound(k, n, alpha):
    """One-sided (1 - alpha) lower bound on a binomial proportion: Clopper-Pearson if scipy is available, else Hoeffding."""
    try:
        from scipy.stats import beta
        return float(beta.ppf(alpha, k, n - k + 1)) if k > 0 else 0.0
    except ImportError:
        return max(0.0, k / n - math.sqrt(math.log(1 / alpha) / (2 * n)))

@torch.no_grad()
def sample_counts(model_, x, sigma, n, batch_size=250):
    model_.eval()
    counts = torch.zeros(10, dtype=torch.long)
    for i in range(0, n, batch_size):
        m = min(batch_size, n - i)
        noisy = (x.repeat(m, 1, 1, 1) + sigma * torch.randn(m, *x.shape[1:], device=x.device)).clamp(0.0, 1.0)
        counts += torch.bincount(model_(noisy).argmax(1).cpu(), minlength=10)
    return counts

def certify(model_, x, sigma, n0=64, n=N_CERT, alpha=0.001):
    """Randomized-smoothing certificate: (predicted class, lower bound on p_A, certified L2 radius or None)."""
    c_A = int(sample_counts(model_, x, sigma, n0).argmax())
    n_A = int(sample_counts(model_, x, sigma, n)[c_A])
    p_lower = lower_confidence_bound(n_A, n, alpha)
    if p_lower <= 0.5:
        return c_A, p_lower, None
    radius = sigma * torch.distributions.Normal(0.0, 1.0).icdf(torch.tensor(p_lower)).item()
    return c_A, p_lower, radius

def format_certificate(result):
    c, p_lower, radius = result
    radius_str = f"{radius:.3f}" if radius is not None else "abstain"
    return f"class {c}  p_A >= {p_lower:.3f}  radius {radius_str:>8s}"

torch.manual_seed(SEED)
print(f"\n{'image':>6s} {'label':>6s} | {'standard model':^40s} | {'noise-trained model':^40s}")
for i in range(8):
    x = X_test[i:i + 1].to(device)
    res_std, res_smooth = certify(model, x, SIGMA), certify(smooth_model, x, SIGMA)
    print(f"{i:6d} {int(y_test[i]):6d} | {format_certificate(res_std):40s} | {format_certificate(res_smooth):40s}")

The certificate is a statement about the *smoothed* classifier, the majority vote under Gaussian noise: for every input where $\underline{p_A} > 1/2$, no perturbation of $L_2$ norm below $R$ can change that vote, with probability $1 - \alpha$ over the sampling. The radius grows with $\sigma$ and with the margin of the vote, and it is capped by the sample size: with $n$ noisy copies that all agree, the Clopper–Pearson lower bound is exactly $\alpha^{1/n}$ (about 0.993 for $n = 1000$ and $\alpha = 0.001$, giving $R \approx 2.5\sigma$), which is why many rows show the same maximal radius. The noise-trained model certifies larger radii, or certifies at all, on the inputs where the standard model's predictions under noise are inconsistent (on MNIST at $\sigma = 0.25$ even the standard model is fairly noise-tolerant, so the difference is modest in the short run and grows with $\sigma$). Look also for rows where a certificate is issued for the *wrong* class: the guarantee is about consistency, not correctness, and a smoothed classifier can be provably stable around a mistake. The trade-off is the same as in Section 4.6 seen from the certified side: a larger $\sigma$ allows larger radii but lowers the base classifier's accuracy, and the confidence bound tightens only slowly with $n$, so certification costs hundreds or thousands of forward passes per input. Note also that $L_2$ radii on $[0, 1]$-scaled pixels are not directly comparable to the $L_\infty$ budgets of Section 4; a radius of 0.5 in $L_2$ corresponds to a much smaller per-pixel change than $\epsilon = 0.1$ in $L_\infty$ spread over all 784 pixels.

## Summary

- Test accuracy answers a narrower question than deployment asks; documented failures (shortcut learning, adversarial stickers, overconfidence under shift, disparate error rates) all occurred in systems that had passed their test sets.
- Gradient saliency, SmoothGrad, Integrated Gradients, occlusion and Grad-CAM answer different questions about a prediction; Integrated Gradients satisfies completeness ($\sum_i \mathrm{IG}_i = F(x) - F(x')$), which doubles as a unit test, and every attribution method should pass the model randomization sanity check before it is trusted.
- Activation maximization reveals what units and classes respond to (and that a network's preferred input is not a natural one); linear probes measure how linearly decodable a quantity is at each depth.
- Adversarial examples follow from the near-linearity of networks in high dimensions: FGSM is the one-step optimum of the linearized loss, PGD its iterated, projected version, and both have targeted variants. Adversarial examples transfer between models, so hidden weights are no defence.
- PGD adversarial training is the reliable defence; it costs clean accuracy (the robustness–accuracy trade-off) and yields models with cleaner input gradients. Certified defences (bound propagation, randomized smoothing) give guarantees at a further cost.
- Under distribution shift accuracy falls faster than confidence; OOD inputs can be detected with the maximum softmax probability, better with the energy score, and often better still with feature-space or disagreement-based scores.
- Calibration is measured with reliability diagrams and ECE (and NLL); temperature scaling, fitted on validation logits, fixes in-distribution calibration without changing predictions but does not transfer to shifted data.
- Epistemic uncertainty requires several predictors: MC dropout and, more strongly, deep ensembles improve calibration under shift and OOD detection, with mutual information isolating the disagreement.
- Demographic parity and equalized odds are computable, incompatible criteria; fairness through unawareness fails when proxies exist; post-processing thresholds are a simple mitigation; differential privacy (DP-SGD) bounds what a model can reveal about any one record.
- Datasheets, model cards and a deployment checklist turn these measurements into decisions that others can inspect.

## Further reading

- Simonyan, K., Vedaldi, A. and Zisserman, A., 2014, Deep Inside Convolutional Networks: Visualising Image Classification Models and Saliency Maps, *ICLR Workshop*; Smilkov, D. et al., 2017, SmoothGrad: removing noise by adding noise, arXiv:1706.03825.
- Sundararajan, M., Taly, A. and Yan, Q., 2017, Axiomatic Attribution for Deep Networks, *ICML*. Integrated Gradients and its axioms.
- Selvaraju, R. R. et al., 2017, Grad-CAM: Visual Explanations from Deep Networks via Gradient-based Localization, *ICCV*; Zeiler, M. D. and Fergus, R., 2014, Visualizing and Understanding Convolutional Networks, *ECCV*.
- Adebayo, J. et al., 2018, Sanity Checks for Saliency Maps, *NeurIPS*. The model randomization test.
- Olah, C., Mordvintsev, A. and Schubert, L., 2017, Feature Visualization, *Distill*, https://distill.pub/2017/feature-visualization/; Alain, G. and Bengio, Y., 2017, Understanding Intermediate Layers Using Linear Classifier Probes, *ICLR Workshop*; Kim, B. et al., 2018, Interpretability Beyond Feature Attribution: Quantitative Testing with Concept Activation Vectors (TCAV), *ICML*.
- Goodfellow, I., Shlens, J. and Szegedy, C., 2015, Explaining and Harnessing Adversarial Examples, *ICLR*; Madry, A. et al., 2018, Towards Deep Learning Models Resistant to Adversarial Attacks, *ICLR*; Carlini, N. and Wagner, D., 2017, Towards Evaluating the Robustness of Neural Networks, *IEEE S&P*; Athalye, A., Carlini, N. and Wagner, D., 2018, Obfuscated Gradients Give a False Sense of Security, *ICML*; Cohen, J., Rosenfeld, E. and Kolter, J. Z., 2019, Certified Adversarial Robustness via Randomized Smoothing, *ICML*.
- Hendrycks, D. and Gimpel, K., 2017, A Baseline for Detecting Misclassified and Out-of-Distribution Examples in Neural Networks, *ICLR*; Liu, W. et al., 2020, Energy-based Out-of-distribution Detection, *NeurIPS*; Lee, K. et al., 2018, A Simple Unified Framework for Detecting Out-of-Distribution Samples and Adversarial Attacks, *NeurIPS*.
- Guo, C. et al., 2017, On Calibration of Modern Neural Networks, *ICML*; Gal, Y. and Ghahramani, Z., 2016, Dropout as a Bayesian Approximation, *ICML*; Lakshminarayanan, B., Pritzel, A. and Blundell, C., 2017, Simple and Scalable Predictive Uncertainty Estimation using Deep Ensembles, *NeurIPS*; Ovadia, Y. et al., 2019, Can You Trust Your Model's Uncertainty? Evaluating Predictive Uncertainty Under Dataset Shift, *NeurIPS*.
- Hardt, M., Price, E. and Srebro, N., 2016, Equality of Opportunity in Supervised Learning, *NeurIPS*; Chouldechova, A., 2017, Fair Prediction with Disparate Impact, *Big Data*; Barocas, S., Hardt, M. and Narayanan, A., *Fairness and Machine Learning: Limitations and Opportunities*, MIT Press, 2023, https://fairmlbook.org/
- Abadi, M. et al., 2016, Deep Learning with Differential Privacy, *CCS*; Dwork, C. and Roth, A., 2014, The Algorithmic Foundations of Differential Privacy.
- Gebru, T. et al., 2021, Datasheets for Datasets, *Communications of the ACM*; Mitchell, M. et al., 2019, Model Cards for Model Reporting, *FAT\**; Buolamwini, J. and Gebru, T., 2018, Gender Shades, *FAT\**; Geirhos, R. et al., 2020, Shortcut Learning in Deep Neural Networks, *Nature Machine Intelligence*.
- Molnar, C., *Interpretable Machine Learning* (online book), https://christophm.github.io/interpretable-ml-book/